In [ ]:
"""
Clean CIFAR-10 Baseline Training Code

Goal:
- Train a ResNet-18 classifier on clean CIFAR-10 training data only.
- Use a fixed train/calibration split:
    - 45,000 samples for classifier training
    - 5,000 samples reserved for calibration
- Save the trained model to:
    ./results/cifar10_resnet18_clean.pth

Important:
- No OOD data is used during training.
- This saved model will be used later for contaminated calibration experiments.
"""

import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic CuDNN behavior.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"
MODEL_PATH = os.path.join(RESULT_DIR, "cifar10_resnet18_clean.pth")

BATCH_SIZE = 128
NUM_EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 5e-4
NUM_CLASSES = 10

# The same split should be used later in calibration experiments.
TRAIN_SIZE = 45000
CALIB_SIZE = 5000

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

"""
CIFAR-10 is used as the in-distribution dataset.

Split:
- 45,000 samples: classifier training
- 5,000 samples : reserved calibration split

Only the training split is used in this script.
The calibration split is reserved for later threshold estimation.
"""

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# Load CIFAR-10 training data with augmentation.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform
)

# Create the same train/calibration split used in later experiments.
train_dataset, calib_dataset_unused = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

# CIFAR-10 test set is used only for checking classification accuracy.
id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model definition
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This architecture must match the model definition used in later
    OOD experiments, because the saved state_dict will be loaded there.

    The model exposes:
    - logits for classification
    - penultimate features for KNN and ViM in later experiments
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        # Modify the first convolution for 32x32 CIFAR-10 images.
        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # Remove the initial max pooling layer for small images.
        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features before the classifier head.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True, return both logits and penultimate features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)


# =========================================================
# 5. Training function
# =========================================================

def train_classifier(model, train_loader, num_epochs):
    """
    Train the classifier using only clean CIFAR-10 training data.
    """
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )

    for epoch in range(num_epochs):
        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        avg_loss = total_loss / total
        train_acc = correct / total * 100.0

        print(
            f"[Epoch {epoch + 1:03d}/{num_epochs}] "
            f"Loss: {avg_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}%"
        )


# =========================================================
# 6. Evaluation function
# =========================================================

@torch.no_grad()
def evaluate_accuracy(model, data_loader):
    """
    Evaluate classification accuracy on the CIFAR-10 test set.
    """
    model.eval()

    correct = 0
    total = 0

    for images, labels in data_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        preds = logits.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total * 100.0

    return acc


# =========================================================
# 7. Main
# =========================================================

def main():
    """
    Train the clean CIFAR-10 classifier and save the model.
    """

    print("\n[Step 1] Train clean CIFAR-10 classifier")
    train_classifier(
        model=model,
        train_loader=train_loader,
        num_epochs=NUM_EPOCHS
    )

    print("\n[Step 2] Evaluate CIFAR-10 test accuracy")
    test_acc = evaluate_accuracy(
        model=model,
        data_loader=id_test_loader
    )

    print(f"CIFAR-10 Test Accuracy: {test_acc:.2f}%")

    print("\n[Step 3] Save trained model")

    torch.save(
        model.state_dict(),
        MODEL_PATH
    )

    print(f"Saved model to: {MODEL_PATH}")

    # Optional: save a small metadata file.
    metadata = {
        "model_path": MODEL_PATH,
        "dataset": "CIFAR-10",
        "train_size": TRAIN_SIZE,
        "calib_size": CALIB_SIZE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "test_accuracy": test_acc
    }

    metadata_path = os.path.join(RESULT_DIR, "clean_training_metadata.npy")

    np.save(
        metadata_path,
        metadata,
        allow_pickle=True
    )

    print(f"Saved metadata to: {metadata_path}")


if __name__ == "__main__":
    main()

Using device: cuda

[Step 1] Train clean CIFAR-10 classifier
[Epoch 001/50] Loss: 1.4213 | Train Acc: 47.87%
[Epoch 002/50] Loss: 0.9984 | Train Acc: 64.45%
[Epoch 003/50] Loss: 0.8282 | Train Acc: 71.02%
[Epoch 004/50] Loss: 0.7266 | Train Acc: 74.90%
[Epoch 005/50] Loss: 0.6521 | Train Acc: 77.41%
[Epoch 006/50] Loss: 0.5962 | Train Acc: 79.56%
[Epoch 007/50] Loss: 0.5481 | Train Acc: 81.48%
[Epoch 008/50] Loss: 0.5107 | Train Acc: 82.80%
[Epoch 009/50] Loss: 0.4739 | Train Acc: 83.96%
[Epoch 010/50] Loss: 0.4515 | Train Acc: 84.51%
[Epoch 011/50] Loss: 0.4227 | Train Acc: 85.46%
[Epoch 012/50] Loss: 0.3925 | Train Acc: 86.80%
[Epoch 013/50] Loss: 0.3784 | Train Acc: 87.08%
[Epoch 014/50] Loss: 0.3606 | Train Acc: 87.79%
[Epoch 015/50] Loss: 0.3428 | Train Acc: 88.34%
[Epoch 016/50] Loss: 0.3257 | Train Acc: 89.04%
[Epoch 017/50] Loss: 0.3093 | Train Acc: 89.44%
[Epoch 018/50] Loss: 0.2947 | Train Acc: 89.95%
[Epoch 019/50] Loss: 0.2812 | Train Acc: 90.39%
[Epoch 020/50] Loss: 0.2698

# Contaminated Calibration Set

In [ ]:
"""
Contaminated Calibration Experiment for Post-hoc OOD Detection

Goal:
- Use the already trained CIFAR-10 classifier.
- Keep the train set and test sets fixed.
- Change only the calibration set by injecting OOD samples.
- Evaluate how FPR95 changes as calibration contamination increases.

Protocol:
- ID dataset        : CIFAR-10
- OOD dataset       : SVHN
- Train set         : CIFAR-10 train only
- ID test set       : CIFAR-10 test
- OOD test set      : SVHN test
- Contamination OOD : SVHN train subset
- Calibration set   : CIFAR-10 calibration split + SVHN train subset

Contamination ratios:
- 0%, 1%, 5%, 10%, 20%

OOD methods:
1. MSP
2. Energy
3. KNN
4. ViM
5. ODIN

Important:
- The classifier is NOT retrained.
- The ID test set and OOD test set are NOT changed.
- Only the calibration set is changed.
- AUROC is expected to remain almost unchanged.
- FPR95 and threshold are the main metrics of interest.
"""

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Basic configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"

MODEL_PATH = "./results/cifar10_resnet18_clean.pth"

BATCH_SIZE = 128
NUM_CLASSES = 10

TRAIN_SIZE = 45000
CALIB_SIZE = 5000

ENERGY_T = 1.0

KNN_K = 50

VIM_DIM = 256

ODIN_T = 1000.0
ODIN_EPSILON = 0.0014

CONTAMINATION_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20]

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

"""
Dataset usage:

CIFAR-10 train:
- Used to reconstruct the same train/calibration split.
- Train split is used only for feature bank fitting in KNN and ViM.
- Calibration split is used as clean ID calibration data.

SVHN train:
- Used only as the source of OOD contamination for the calibration set.

CIFAR-10 test:
- Used as ID test data.

SVHN test:
- Used as OOD test data.
"""

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# Load CIFAR-10 training data with deterministic transform.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=test_transform
)

# Reconstruct the same train/calibration split used in clean baseline.
train_dataset, clean_calib_dataset = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

# CIFAR-10 test set for ID testing.
id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

# SVHN train set is used as contamination source.
svhn_train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=test_transform
)

# SVHN test set is used as final OOD test set.
ood_test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ood_test_loader = DataLoader(
    ood_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model definition
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This model returns:
    - logits for MSP, Energy, and ODIN
    - penultimate features for KNN and ViM
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features before the final classifier.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True, return both logits and features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)

# Load the already trained clean CIFAR-10 classifier.
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 5. Utility functions
# =========================================================

def create_contaminated_calibration_dataset(
    clean_calib_dataset,
    svhn_train_dataset,
    contamination_ratio,
    total_calib_size=5000,
    seed=42
):
    """
    Create a contaminated calibration set.

    The total calibration size is fixed.

    Example:
    - contamination_ratio = 0.05
    - total_calib_size = 5000
    - ID samples  = 4750
    - OOD samples = 250

    This ensures that the calibration set size does not change across ratios.
    Therefore, any threshold shift comes from contamination, not sample size.
    """
    rng = np.random.default_rng(seed)

    num_ood = int(total_calib_size * contamination_ratio)
    num_id = total_calib_size - num_ood

    # Sample ID calibration indices from clean CIFAR-10 calibration split.
    id_indices = rng.choice(
        len(clean_calib_dataset),
        size=num_id,
        replace=False
    )

    # Sample OOD contamination indices from SVHN train split.
    ood_indices = rng.choice(
        len(svhn_train_dataset),
        size=num_ood,
        replace=False
    )

    id_subset = Subset(clean_calib_dataset, id_indices.tolist())
    ood_subset = Subset(svhn_train_dataset, ood_indices.tolist())

    contaminated_calib_dataset = ConcatDataset([id_subset, ood_subset])

    return contaminated_calib_dataset, num_id, num_ood


@torch.no_grad()
def extract_logits_and_features(model, data_loader):
    """
    Extract logits and penultimate features from a dataloader.
    """
    model.eval()

    all_logits = []
    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(DEVICE)

        logits, features = model(images, return_features=True)

        all_logits.append(logits.cpu())
        all_features.append(features.cpu())
        all_labels.append(torch.as_tensor(labels))

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_features = torch.cat(all_features, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    return all_logits, all_features, all_labels


def compute_msp_from_logits(logits):
    """
    Compute Maximum Softmax Probability.

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    probs = torch.softmax(logits_tensor, dim=1)
    scores = probs.max(dim=1).values.numpy()

    return scores


def compute_energy_from_logits(logits, temperature=1.0):
    """
    Compute Energy-based ID score.

    This returns -E(x), so higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    scores = temperature * torch.logsumexp(
        logits_tensor / temperature,
        dim=1
    ).numpy()

    return scores


def fit_knn_feature_bank(train_features, k=50):
    """
    Fit KNN using clean ID training features.
    """
    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=-1
    )

    knn.fit(train_features)

    return knn


def compute_knn_scores(knn, features, k=50):
    """
    Compute KNN-based ID scores.

    The negative distance to the k-th nearest neighbor is used as the ID score.
    Higher score means more ID-like.
    """
    distances, _ = knn.kneighbors(features, n_neighbors=k)
    kth_distance = distances[:, -1]

    scores = -kth_distance

    return scores


def fit_vim(train_features, train_logits, vim_dim=256):
    """
    Fit simplified ViM statistics using clean ID training features.
    """
    feature_mean = np.mean(train_features, axis=0, keepdims=True)
    centered_features = train_features - feature_mean

    _, _, vt = np.linalg.svd(centered_features, full_matrices=False)

    feature_dim = train_features.shape[1]
    vim_dim = min(vim_dim, feature_dim - 1)

    residual_space = vt[vim_dim:].T

    train_residual = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    train_max_logit = np.max(train_logits, axis=1)

    alpha = np.mean(train_max_logit) / (np.mean(train_residual) + 1e-12)

    vim_stats = {
        "feature_mean": feature_mean,
        "residual_space": residual_space,
        "alpha": alpha,
        "vim_dim": vim_dim
    }

    return vim_stats


def compute_vim_scores(features, logits, vim_stats):
    """
    Compute simplified ViM ID scores.

    score = max_original_logit - virtual_logit

    Higher score means more ID-like.
    """
    feature_mean = vim_stats["feature_mean"]
    residual_space = vim_stats["residual_space"]
    alpha = vim_stats["alpha"]

    centered_features = features - feature_mean

    residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    virtual_logit = alpha * residual_norm
    max_original_logit = np.max(logits, axis=1)

    scores = max_original_logit - virtual_logit

    return scores


def compute_odin_scores(model, data_loader, temperature=1000.0, epsilon=0.0014):
    """
    Compute ODIN scores.

    ODIN requires gradient computation at inference time.

    Higher score means more ID-like.
    """
    model.eval()

    all_scores = []

    for images, _ in data_loader:
        images = images.to(DEVICE)
        images.requires_grad_(True)

        logits = model(images)
        scaled_logits = logits / temperature

        pseudo_labels = scaled_logits.argmax(dim=1)
        loss = F.cross_entropy(scaled_logits, pseudo_labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = torch.sign(images.grad.data)

        perturbed_images = images - epsilon * gradient_sign
        perturbed_images = perturbed_images.detach()

        with torch.no_grad():
            perturbed_logits = model(perturbed_images)
            perturbed_scaled_logits = perturbed_logits / temperature
            probs = torch.softmax(perturbed_scaled_logits, dim=1)

            scores = probs.max(dim=1).values

        all_scores.append(scores.cpu())

    all_scores = torch.cat(all_scores, dim=0).numpy()

    return all_scores


def get_threshold_at_95_tpr(calib_scores):
    """
    Estimate threshold from calibration scores.

    Since higher score means more ID-like:
    - The 5th percentile threshold accepts about 95% of calibration samples.
    """
    threshold = np.percentile(calib_scores, 5)

    return threshold


def compute_fpr95(ood_scores, threshold):
    """
    Compute FPR95.

    OOD samples with score >= threshold are incorrectly accepted as ID.
    """
    false_positive = np.sum(ood_scores >= threshold)
    fpr95 = false_positive / len(ood_scores)

    return fpr95


def compute_auroc(id_scores, ood_scores):
    """
    Compute AUROC.

    ID label  = 1
    OOD label = 0
    """
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores))
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores
    ])

    return roc_auc_score(y_true, y_score)


def evaluate_with_threshold(method_name, calib_scores, id_test_scores, ood_test_scores):
    """
    Evaluate one OOD method under a given calibration set.
    """
    threshold = get_threshold_at_95_tpr(calib_scores)

    auroc = compute_auroc(id_test_scores, ood_test_scores)
    fpr95 = compute_fpr95(ood_test_scores, threshold)

    return {
        "method": method_name,
        "threshold": float(threshold),
        "auroc": float(auroc),
        "fpr95": float(fpr95)
    }


# =========================================================
# 6. Main contaminated calibration experiment
# =========================================================

def main():
    """
    Run contaminated calibration experiments.

    The expensive parts are computed once:
    - train logits/features
    - ID test logits/features
    - OOD test logits/features
    - KNN feature bank
    - ViM statistics
    - ID/OOD test scores for each method

    For each contamination ratio:
    - create contaminated calibration set
    - compute calibration scores
    - estimate threshold
    - compute FPR95
    """

    # -----------------------------------------------------
    # Step 1. Extract fixed train/test logits and features
    # -----------------------------------------------------
    print("\n[Step 1] Extract fixed train/test logits and features")

    train_logits, train_features, _ = extract_logits_and_features(
        model,
        train_loader
    )

    id_test_logits, id_test_features, _ = extract_logits_and_features(
        model,
        id_test_loader
    )

    ood_test_logits, ood_test_features, _ = extract_logits_and_features(
        model,
        ood_test_loader
    )

    print(f"Train features shape   : {train_features.shape}")
    print(f"ID test features shape : {id_test_features.shape}")
    print(f"OOD test features shape: {ood_test_features.shape}")

    # -----------------------------------------------------
    # Step 2. Fit KNN and ViM using only clean ID train data
    # -----------------------------------------------------
    print("\n[Step 2] Fit KNN and ViM using clean ID train data")

    knn = fit_knn_feature_bank(
        train_features=train_features,
        k=KNN_K
    )

    vim_stats = fit_vim(
        train_features=train_features,
        train_logits=train_logits,
        vim_dim=VIM_DIM
    )

    # -----------------------------------------------------
    # Step 3. Precompute fixed ID/OOD test scores
    # -----------------------------------------------------
    print("\n[Step 3] Precompute fixed ID/OOD test scores")

    fixed_test_scores = {}

    fixed_test_scores["MSP"] = {
        "id": compute_msp_from_logits(id_test_logits),
        "ood": compute_msp_from_logits(ood_test_logits)
    }

    fixed_test_scores["Energy"] = {
        "id": compute_energy_from_logits(id_test_logits, temperature=ENERGY_T),
        "ood": compute_energy_from_logits(ood_test_logits, temperature=ENERGY_T)
    }

    fixed_test_scores[f"KNN-k{KNN_K}"] = {
        "id": compute_knn_scores(knn, id_test_features, k=KNN_K),
        "ood": compute_knn_scores(knn, ood_test_features, k=KNN_K)
    }

    fixed_test_scores[f"ViM-dim{vim_stats['vim_dim']}"] = {
        "id": compute_vim_scores(id_test_features, id_test_logits, vim_stats),
        "ood": compute_vim_scores(ood_test_features, ood_test_logits, vim_stats)
    }

    # ODIN is slower because it requires gradients.
    fixed_test_scores[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = {
        "id": compute_odin_scores(
            model,
            id_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        ),
        "ood": compute_odin_scores(
            model,
            ood_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )
    }

    # -----------------------------------------------------
    # Step 4. Run each contamination ratio
    # -----------------------------------------------------
    print("\n[Step 4] Run contaminated calibration experiments")

    all_results = []

    for ratio in CONTAMINATION_RATIOS:
        print("\n" + "=" * 80)
        print(f"Contamination ratio: {ratio * 100:.1f}%")
        print("=" * 80)

        contaminated_calib_dataset, num_id, num_ood = create_contaminated_calibration_dataset(
            clean_calib_dataset=clean_calib_dataset,
            svhn_train_dataset=svhn_train_dataset,
            contamination_ratio=ratio,
            total_calib_size=CALIB_SIZE,
            seed=42 + int(ratio * 10000)
        )

        contaminated_calib_loader = DataLoader(
            contaminated_calib_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        print(f"Calibration ID samples : {num_id}")
        print(f"Calibration OOD samples: {num_ood}")

        # Extract logits/features from the current contaminated calibration set.
        calib_logits, calib_features, _ = extract_logits_and_features(
            model,
            contaminated_calib_loader
        )

        # Compute calibration scores for each method.
        calib_scores_dict = {}

        calib_scores_dict["MSP"] = compute_msp_from_logits(calib_logits)

        calib_scores_dict["Energy"] = compute_energy_from_logits(
            calib_logits,
            temperature=ENERGY_T
        )

        calib_scores_dict[f"KNN-k{KNN_K}"] = compute_knn_scores(
            knn,
            calib_features,
            k=KNN_K
        )

        calib_scores_dict[f"ViM-dim{vim_stats['vim_dim']}"] = compute_vim_scores(
            calib_features,
            calib_logits,
            vim_stats
        )

        calib_scores_dict[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = compute_odin_scores(
            model,
            contaminated_calib_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )

        # Evaluate all methods using the threshold estimated from contaminated calibration scores.
        for method_name, calib_scores in calib_scores_dict.items():
            result = evaluate_with_threshold(
                method_name=method_name,
                calib_scores=calib_scores,
                id_test_scores=fixed_test_scores[method_name]["id"],
                ood_test_scores=fixed_test_scores[method_name]["ood"]
            )

            row = {
                "contamination_ratio": ratio,
                "contamination_percent": ratio * 100,
                "num_calib_id": num_id,
                "num_calib_ood": num_ood,
                "method": method_name,
                "threshold": result["threshold"],
                "auroc": result["auroc"],
                "fpr95": result["fpr95"],
                "auroc_percent": result["auroc"] * 100,
                "fpr95_percent": result["fpr95"] * 100
            }

            all_results.append(row)

            print(
                f"{method_name:<25} | "
                f"Threshold: {result['threshold']:.6f} | "
                f"AUROC: {result['auroc'] * 100:6.2f}% | "
                f"FPR95: {result['fpr95'] * 100:6.2f}%"
            )

    # -----------------------------------------------------
    # Step 5. Save results
    # -----------------------------------------------------
    results_df = pd.DataFrame(all_results)

    csv_path = os.path.join(
        RESULT_DIR,
        "contaminated_calibration_results.csv"
    )

    npy_path = os.path.join(
        RESULT_DIR,
        "contaminated_calibration_results.npy"
    )

    results_df.to_csv(csv_path, index=False)
    np.save(npy_path, all_results, allow_pickle=True)

    print("\nSaved contaminated calibration results to:")
    print(csv_path)
    print(npy_path)

    # -----------------------------------------------------
    # Step 6. Print compact FPR95 table
    # -----------------------------------------------------
    print("\n" + "=" * 80)
    print("FPR95 (%) Table")
    print("=" * 80)

    fpr_table = results_df.pivot(
        index="method",
        columns="contamination_percent",
        values="fpr95_percent"
    )

    print(fpr_table.round(2))

    fpr_table_path = os.path.join(
        RESULT_DIR,
        "contaminated_calibration_fpr95_table.csv"
    )

    fpr_table.to_csv(fpr_table_path)

    print("\nSaved FPR95 table to:")
    print(fpr_table_path)


if __name__ == "__main__":
    main()

Using device: cuda
Loaded model from: ./results/cifar10_resnet18_clean.pth

[Step 1] Extract fixed train/test logits and features
Train features shape   : (45000, 512)
ID test features shape : (10000, 512)
OOD test features shape: (26032, 512)

[Step 2] Fit KNN and ViM using clean ID train data

[Step 3] Precompute fixed ID/OOD test scores

[Step 4] Run contaminated calibration experiments

Contamination ratio: 0.0%
Calibration ID samples : 5000
Calibration OOD samples: 0
MSP                       | Threshold: 0.774561 | AUROC:  90.77% | FPR95:  59.37%
Energy                    | Threshold: 4.647542 | AUROC:  91.55% | FPR95:  53.74%
KNN-k50                   | Threshold: -3.273125 | AUROC:  92.84% | FPR95:  46.30%
ViM-dim256                | Threshold: -8.692730 | AUROC:  95.53% | FPR95:  25.82%
ODIN-T1000.0-eps0.0014    | Threshold: 0.100733 | AUROC:  91.32% | FPR95:  53.40%

Contamination ratio: 1.0%
Calibration ID samples : 4950
Calibration OOD samples: 50
MSP                       

# Robust Trimming Calibration

In [ ]:
"""
Robust Trimming Calibration Experiment for Post-hoc OOD Detection

Goal:
- Use a pretrained CIFAR-10 classifier.
- Keep the classifier fixed.
- Keep ID test and OOD test sets fixed.
- Contaminate only the calibration set with SVHN samples.
- Compare standard calibration and robust trimming calibration.

Dataset protocol:
- ID dataset        : CIFAR-10
- OOD dataset       : SVHN
- Train feature bank: CIFAR-10 train split only
- Clean calibration : CIFAR-10 calibration split only
- Contamination OOD : SVHN train split
- ID test           : CIFAR-10 test
- OOD test          : SVHN test

OOD methods:
1. MSP
2. Energy
3. KNN
4. ViM
5. ODIN

Calibration methods:
1. Standard calibration
   - Use all calibration scores.
   - Threshold = 5th percentile of calibration scores.

2. Robust trimming calibration
   - Remove the lowest q% calibration scores.
   - Then compute the 5th percentile threshold from the remaining scores.

Score convention:
- Higher score = more ID-like
- Lower score  = more OOD-like

Therefore:
- threshold = 5th percentile
- score >= threshold -> ID
- score < threshold  -> OOD
"""

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset

from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"

MODEL_PATH = "./results/cifar10_resnet18_clean.pth"

BATCH_SIZE = 128
NUM_CLASSES = 10

# These must match the split used in the clean baseline experiment.
TRAIN_SIZE = 45000
CALIB_SIZE = 5000

# OOD method hyperparameters.
ENERGY_T = 1.0
KNN_K = 50
VIM_DIM = 256

# ODIN baseline hyperparameters.
# These can be tuned later, but fixed values are used here for consistency.
ODIN_T = 1000.0
ODIN_EPSILON = 0.0014

# Calibration contamination ratios.
CONTAMINATION_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20]

# Robust trimming ratios.
# 0.0 corresponds to standard calibration.
TRIM_RATIOS = [0.0, 0.01, 0.05, 0.10]

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

"""
Important:
- The model was trained on CIFAR-10 normalized images.
- Therefore, the same CIFAR-10 normalization is applied to both ID and OOD data.
"""

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# CIFAR-10 train data is used to reconstruct the same train/calibration split.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=test_transform
)

# Reconstruct the same split:
# - train_dataset: used for KNN feature bank and ViM statistics
# - clean_calib_dataset: used as clean ID calibration pool
train_dataset, clean_calib_dataset = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

# CIFAR-10 test set is the ID test set.
id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

# SVHN train set is used as contamination source for calibration.
svhn_train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=test_transform
)

# SVHN test set is used only as final OOD test set.
ood_test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ood_test_loader = DataLoader(
    ood_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This wrapper exposes:
    - logits for MSP, Energy, and ODIN
    - penultimate features for KNN and ViM
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        # CIFAR-10 images are 32x32, so the ImageNet-style first layer is modified.
        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # Remove initial max pooling for small images.
        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True:
            return logits and penultimate features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}\n"
        "Please run the clean baseline training script first."
    )

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 5. Dataset construction for contaminated calibration
# =========================================================

def create_contaminated_calibration_dataset(
    clean_calib_dataset,
    svhn_train_dataset,
    contamination_ratio,
    total_calib_size=5000,
    seed=42
):
    """
    Create a contaminated calibration dataset.

    The total calibration size is fixed for all contamination ratios.

    Example:
    - total_calib_size = 5000
    - contamination_ratio = 0.05
    - ID samples  = 4750
    - OOD samples = 250

    This is important because we want threshold changes to be caused by
    OOD contamination, not by calibration set size changes.
    """
    rng = np.random.default_rng(seed)

    num_ood = int(total_calib_size * contamination_ratio)
    num_id = total_calib_size - num_ood

    id_indices = rng.choice(
        len(clean_calib_dataset),
        size=num_id,
        replace=False
    )

    if num_ood > 0:
        ood_indices = rng.choice(
            len(svhn_train_dataset),
            size=num_ood,
            replace=False
        )

        id_subset = Subset(clean_calib_dataset, id_indices.tolist())
        ood_subset = Subset(svhn_train_dataset, ood_indices.tolist())

        contaminated_dataset = ConcatDataset([id_subset, ood_subset])

    else:
        contaminated_dataset = Subset(clean_calib_dataset, id_indices.tolist())

    return contaminated_dataset, num_id, num_ood


# =========================================================
# 6. Logit and feature extraction
# =========================================================

@torch.no_grad()
def extract_logits_and_features(model, data_loader):
    """
    Extract logits and penultimate features from a dataloader.

    Returns:
    - logits   : shape [N, num_classes]
    - features : shape [N, feature_dim]
    - labels   : shape [N]
    """
    model.eval()

    all_logits = []
    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(DEVICE)

        logits, features = model(images, return_features=True)

        all_logits.append(logits.detach().cpu())
        all_features.append(features.detach().cpu())
        all_labels.append(torch.as_tensor(labels))

    logits = torch.cat(all_logits, dim=0).numpy()
    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    return logits, features, labels


# =========================================================
# 7. OOD score functions
# =========================================================

def compute_msp_from_logits(logits):
    """
    Compute MSP scores.

    MSP = maximum softmax probability.

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    probs = torch.softmax(logits_tensor, dim=1)
    scores = probs.max(dim=1).values.numpy()

    return scores


def compute_energy_from_logits(logits, temperature=1.0):
    """
    Compute Energy-based ID scores.

    Standard energy:
        E(x) = -T * logsumexp(logits / T)

    This function returns -E(x):
        score(x) = T * logsumexp(logits / T)

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    scores = temperature * torch.logsumexp(
        logits_tensor / temperature,
        dim=1
    ).numpy()

    return scores


def fit_knn_feature_bank(train_features, k=50):
    """
    Fit KNN on clean ID training features.

    KNN intuition:
    - ID samples should be close to the ID feature manifold.
    - OOD samples should be far from the ID feature manifold.
    """
    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=-1
    )

    knn.fit(train_features)

    return knn


def compute_knn_scores(knn, features, k=50):
    """
    Compute KNN-based ID scores.

    We use the negative distance to the k-th nearest neighbor.

    Higher score means more ID-like.
    """
    distances, _ = knn.kneighbors(features, n_neighbors=k)

    kth_distance = distances[:, -1]
    scores = -kth_distance

    return scores


def fit_vim(train_features, train_logits, vim_dim=256):
    """
    Fit simplified ViM statistics using clean ID training data.

    Simplified ViM steps:
    1. Estimate the ID feature mean.
    2. Compute principal directions using SVD.
    3. Define the residual subspace as the complement of the principal subspace.
    4. Scale residual norm using alpha.
    """
    feature_mean = np.mean(train_features, axis=0, keepdims=True)

    centered_features = train_features - feature_mean

    _, _, vt = np.linalg.svd(centered_features, full_matrices=False)

    feature_dim = train_features.shape[1]
    vim_dim = min(vim_dim, feature_dim - 1)

    residual_space = vt[vim_dim:].T

    train_residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    train_max_logit = np.max(train_logits, axis=1)

    alpha = np.mean(train_max_logit) / (np.mean(train_residual_norm) + 1e-12)

    vim_stats = {
        "feature_mean": feature_mean,
        "residual_space": residual_space,
        "alpha": alpha,
        "vim_dim": vim_dim
    }

    return vim_stats


def compute_vim_scores(features, logits, vim_stats):
    """
    Compute simplified ViM ID scores.

    virtual_logit = alpha * residual_norm

    score = max_original_logit - virtual_logit

    Higher score means more ID-like.
    """
    feature_mean = vim_stats["feature_mean"]
    residual_space = vim_stats["residual_space"]
    alpha = vim_stats["alpha"]

    centered_features = features - feature_mean

    residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    virtual_logit = alpha * residual_norm
    max_original_logit = np.max(logits, axis=1)

    scores = max_original_logit - virtual_logit

    return scores


def compute_odin_scores(model, data_loader, temperature=1000.0, epsilon=0.0014):
    """
    Compute ODIN scores.

    ODIN uses:
    1. Temperature scaling
    2. Small input perturbation based on gradients

    This function returns the maximum softmax probability after ODIN processing.

    Higher score means more ID-like.
    """
    model.eval()

    all_scores = []

    for images, _ in data_loader:
        images = images.to(DEVICE)
        images.requires_grad_(True)

        logits = model(images)
        scaled_logits = logits / temperature

        # Use model predictions as pseudo-labels.
        pseudo_labels = scaled_logits.argmax(dim=1)

        loss = F.cross_entropy(scaled_logits, pseudo_labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = torch.sign(images.grad.data)

        # Apply small perturbation.
        perturbed_images = images - epsilon * gradient_sign
        perturbed_images = perturbed_images.detach()

        with torch.no_grad():
            perturbed_logits = model(perturbed_images)
            perturbed_scaled_logits = perturbed_logits / temperature
            probs = torch.softmax(perturbed_scaled_logits, dim=1)

            scores = probs.max(dim=1).values

        all_scores.append(scores.detach().cpu())

    scores = torch.cat(all_scores, dim=0).numpy()

    return scores


# =========================================================
# 8. Threshold and metric functions
# =========================================================

def get_standard_threshold_at_95_tpr(calib_scores):
    """
    Estimate the standard threshold from calibration scores.

    Since higher score means more ID-like:
    - Use the 5th percentile to accept approximately 95% of calibration samples.
    """
    threshold = np.percentile(calib_scores, 5)

    return threshold


def get_trimmed_threshold_at_95_tpr(calib_scores, trim_ratio=0.0):
    """
    Estimate a robust threshold using lower-tail trimming.

    Motivation:
    - Hidden OOD samples in the calibration set tend to have low scores.
    - These low scores pull the 5th percentile threshold downward.
    - A lower threshold makes OOD rejection weaker and increases FPR95.

    Procedure:
    1. Sort calibration scores in ascending order.
    2. Remove the lowest trim_ratio portion.
    3. Compute the 5th percentile threshold from the remaining scores.

    trim_ratio = 0.0 is equivalent to standard calibration.
    """
    calib_scores = np.asarray(calib_scores)

    if trim_ratio < 0.0 or trim_ratio >= 1.0:
        raise ValueError("trim_ratio must be in [0.0, 1.0).")

    sorted_scores = np.sort(calib_scores)

    num_samples = len(sorted_scores)
    num_trim = int(num_samples * trim_ratio)

    trimmed_scores = sorted_scores[num_trim:]

    if len(trimmed_scores) == 0:
        raise ValueError("All calibration samples were trimmed.")

    threshold = np.percentile(trimmed_scores, 5)

    return threshold


def compute_fpr95(ood_scores, threshold):
    """
    Compute FPR95.

    FPR95:
    - False positive rate of OOD samples
    - when ID true positive rate is fixed near 95%

    Here:
    - score >= threshold means accepted as ID.
    - OOD samples with score >= threshold are false positives.
    """
    false_positive = np.sum(ood_scores >= threshold)
    fpr95 = false_positive / len(ood_scores)

    return fpr95


def compute_auroc(id_scores, ood_scores):
    """
    Compute AUROC.

    Label convention:
    - ID  = 1
    - OOD = 0

    Score convention:
    - Higher score means more ID-like.
    """
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores))
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores
    ])

    auroc = roc_auc_score(y_true, y_score)

    return auroc


def evaluate_ood_method(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores,
    trim_ratio
):
    """
    Evaluate one OOD method under one calibration setting.

    trim_ratio:
    - 0.0 means standard calibration.
    - >0.0 means robust lower-tail trimming calibration.
    """
    threshold = get_trimmed_threshold_at_95_tpr(
        calib_scores=calib_scores,
        trim_ratio=trim_ratio
    )

    auroc = compute_auroc(
        id_scores=id_test_scores,
        ood_scores=ood_test_scores
    )

    fpr95 = compute_fpr95(
        ood_scores=ood_test_scores,
        threshold=threshold
    )

    calibration_type = "standard" if trim_ratio == 0.0 else "trimmed"

    result = {
        "method": method_name,
        "calibration_type": calibration_type,
        "trim_ratio": float(trim_ratio),
        "trim_percent": float(trim_ratio * 100),
        "threshold": float(threshold),
        "auroc": float(auroc),
        "fpr95": float(fpr95),
        "auroc_percent": float(auroc * 100),
        "fpr95_percent": float(fpr95 * 100)
    }

    return result


# =========================================================
# 9. Plotting functions
# =========================================================

def save_fpr95_plots(results_df, result_dir):
    """
    Save FPR95 plots for each trimming ratio.

    Each plot:
    - x-axis: contamination ratio
    - y-axis: FPR95
    - one line per OOD method
    """
    for trim_ratio in sorted(results_df["trim_ratio"].unique()):
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        plt.figure(figsize=(8, 5))

        for method in sorted(subset["method"].unique()):
            method_df = subset[subset["method"] == method].sort_values(
                by="contamination_percent"
            )

            plt.plot(
                method_df["contamination_percent"],
                method_df["fpr95_percent"],
                marker="o",
                label=method
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"FPR95 under calibration contamination | trim={trim_ratio * 100:.1f}%")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        save_path = os.path.join(
            result_dir,
            f"fpr95_plot_trim_{int(trim_ratio * 100)}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_method_comparison_plots(results_df, result_dir):
    """
    Save one plot per OOD method.

    Each plot:
    - x-axis: contamination ratio
    - y-axis: FPR95
    - one line per trimming ratio
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 5))

        for trim_ratio in sorted(subset["trim_ratio"].unique()):
            trim_df = subset[subset["trim_ratio"] == trim_ratio].sort_values(
                by="contamination_percent"
            )

            label = "standard" if trim_ratio == 0.0 else f"trim {trim_ratio * 100:.1f}%"

            plt.plot(
                trim_df["contamination_percent"],
                trim_df["fpr95_percent"],
                marker="o",
                label=label
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"Robust calibration effect | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_method_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(
            result_dir,
            f"robust_effect_{safe_method_name}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


# =========================================================
# 10. Main experiment
# =========================================================

def main():
    """
    Run the full robust trimming calibration experiment.

    Main pipeline:
    1. Extract fixed train/test logits and features.
    2. Fit KNN and ViM using clean ID training data.
    3. Precompute fixed ID/OOD test scores for all methods.
    4. For each contamination ratio:
        a. Create contaminated calibration set.
        b. Compute calibration scores.
        c. Estimate thresholds with different trim ratios.
        d. Compute AUROC and FPR95.
    5. Save result tables and plots.
    """

    # -----------------------------------------------------
    # Step 1. Extract fixed train/test logits and features
    # -----------------------------------------------------
    print("\n[Step 1] Extract fixed train/test logits and features")

    train_logits, train_features, _ = extract_logits_and_features(
        model,
        train_loader
    )

    id_test_logits, id_test_features, _ = extract_logits_and_features(
        model,
        id_test_loader
    )

    ood_test_logits, ood_test_features, _ = extract_logits_and_features(
        model,
        ood_test_loader
    )

    print(f"Train logits shape     : {train_logits.shape}")
    print(f"Train features shape   : {train_features.shape}")
    print(f"ID test logits shape   : {id_test_logits.shape}")
    print(f"ID test features shape : {id_test_features.shape}")
    print(f"OOD test logits shape  : {ood_test_logits.shape}")
    print(f"OOD test features shape: {ood_test_features.shape}")

    # -----------------------------------------------------
    # Step 2. Fit KNN and ViM using clean ID training data
    # -----------------------------------------------------
    print("\n[Step 2] Fit KNN and ViM using clean ID training features")

    knn = fit_knn_feature_bank(
        train_features=train_features,
        k=KNN_K
    )

    vim_stats = fit_vim(
        train_features=train_features,
        train_logits=train_logits,
        vim_dim=VIM_DIM
    )

    print(f"KNN k: {KNN_K}")
    print(f"ViM residual start dimension: {vim_stats['vim_dim']}")

    # -----------------------------------------------------
    # Step 3. Precompute fixed ID/OOD test scores
    # -----------------------------------------------------
    print("\n[Step 3] Precompute fixed ID/OOD test scores")

    fixed_test_scores = {}

    fixed_test_scores["MSP"] = {
        "id": compute_msp_from_logits(id_test_logits),
        "ood": compute_msp_from_logits(ood_test_logits)
    }

    fixed_test_scores["Energy"] = {
        "id": compute_energy_from_logits(
            id_test_logits,
            temperature=ENERGY_T
        ),
        "ood": compute_energy_from_logits(
            ood_test_logits,
            temperature=ENERGY_T
        )
    }

    fixed_test_scores[f"KNN-k{KNN_K}"] = {
        "id": compute_knn_scores(
            knn=knn,
            features=id_test_features,
            k=KNN_K
        ),
        "ood": compute_knn_scores(
            knn=knn,
            features=ood_test_features,
            k=KNN_K
        )
    }

    fixed_test_scores[f"ViM-dim{vim_stats['vim_dim']}"] = {
        "id": compute_vim_scores(
            features=id_test_features,
            logits=id_test_logits,
            vim_stats=vim_stats
        ),
        "ood": compute_vim_scores(
            features=ood_test_features,
            logits=ood_test_logits,
            vim_stats=vim_stats
        )
    }

    print("Computing ODIN ID/OOD test scores. This may take longer.")

    fixed_test_scores[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = {
        "id": compute_odin_scores(
            model=model,
            data_loader=id_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        ),
        "ood": compute_odin_scores(
            model=model,
            data_loader=ood_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )
    }

    # -----------------------------------------------------
    # Step 4. Run contaminated calibration experiments
    # -----------------------------------------------------
    print("\n[Step 4] Run contaminated calibration experiments")

    all_results = []

    for contamination_ratio in CONTAMINATION_RATIOS:
        print("\n" + "=" * 100)
        print(f"Contamination ratio: {contamination_ratio * 100:.1f}%")
        print("=" * 100)

        contaminated_calib_dataset, num_id, num_ood = create_contaminated_calibration_dataset(
            clean_calib_dataset=clean_calib_dataset,
            svhn_train_dataset=svhn_train_dataset,
            contamination_ratio=contamination_ratio,
            total_calib_size=CALIB_SIZE,
            seed=42 + int(contamination_ratio * 10000)
        )

        contaminated_calib_loader = DataLoader(
            contaminated_calib_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        print(f"Calibration ID samples : {num_id}")
        print(f"Calibration OOD samples: {num_ood}")

        # Extract logits/features from the current calibration set.
        calib_logits, calib_features, _ = extract_logits_and_features(
            model,
            contaminated_calib_loader
        )

        # Compute calibration scores for all OOD methods.
        calib_scores_dict = {}

        calib_scores_dict["MSP"] = compute_msp_from_logits(calib_logits)

        calib_scores_dict["Energy"] = compute_energy_from_logits(
            calib_logits,
            temperature=ENERGY_T
        )

        calib_scores_dict[f"KNN-k{KNN_K}"] = compute_knn_scores(
            knn=knn,
            features=calib_features,
            k=KNN_K
        )

        calib_scores_dict[f"ViM-dim{vim_stats['vim_dim']}"] = compute_vim_scores(
            features=calib_features,
            logits=calib_logits,
            vim_stats=vim_stats
        )

        print("Computing ODIN calibration scores. This may take longer.")

        calib_scores_dict[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = compute_odin_scores(
            model=model,
            data_loader=contaminated_calib_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )

        # Evaluate each method under each trimming ratio.
        for method_name, calib_scores in calib_scores_dict.items():
            for trim_ratio in TRIM_RATIOS:

                result = evaluate_ood_method(
                    method_name=method_name,
                    calib_scores=calib_scores,
                    id_test_scores=fixed_test_scores[method_name]["id"],
                    ood_test_scores=fixed_test_scores[method_name]["ood"],
                    trim_ratio=trim_ratio
                )

                row = {
                    "contamination_ratio": float(contamination_ratio),
                    "contamination_percent": float(contamination_ratio * 100),
                    "num_calib_id": int(num_id),
                    "num_calib_ood": int(num_ood),
                    **result
                }

                all_results.append(row)

                print(
                    f"{method_name:<28} | "
                    f"calib={result['calibration_type']:<8} | "
                    f"trim={trim_ratio * 100:>5.1f}% | "
                    f"threshold={result['threshold']:>10.6f} | "
                    f"AUROC={result['auroc_percent']:>6.2f}% | "
                    f"FPR95={result['fpr95_percent']:>6.2f}%"
                )

    # -----------------------------------------------------
    # Step 5. Save full results
    # -----------------------------------------------------
    results_df = pd.DataFrame(all_results)

    full_csv_path = os.path.join(
        RESULT_DIR,
        "robust_trimming_calibration_full_results.csv"
    )

    full_npy_path = os.path.join(
        RESULT_DIR,
        "robust_trimming_calibration_full_results.npy"
    )

    results_df.to_csv(full_csv_path, index=False)
    np.save(full_npy_path, all_results, allow_pickle=True)

    print("\nSaved full results to:")
    print(full_csv_path)
    print(full_npy_path)

    # -----------------------------------------------------
    # Step 6. Save FPR95 tables by trim ratio
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("FPR95 (%) tables by trim ratio")
    print("=" * 100)

    for trim_ratio in TRIM_RATIOS:
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        fpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="fpr95_percent"
        )

        print("\n" + "-" * 100)
        print(f"Trim ratio: {trim_ratio * 100:.1f}%")
        print("-" * 100)
        print(fpr_table.round(2))

        table_path = os.path.join(
            RESULT_DIR,
            f"fpr95_table_trim_{int(trim_ratio * 100)}.csv"
        )

        fpr_table.to_csv(table_path)

        print(f"Saved table: {table_path}")

    # -----------------------------------------------------
    # Step 7. Save threshold tables by trim ratio
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("Threshold tables by trim ratio")
    print("=" * 100)

    for trim_ratio in TRIM_RATIOS:
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        threshold_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="threshold"
        )

        print("\n" + "-" * 100)
        print(f"Trim ratio: {trim_ratio * 100:.1f}%")
        print("-" * 100)
        print(threshold_table.round(6))

        table_path = os.path.join(
            RESULT_DIR,
            f"threshold_table_trim_{int(trim_ratio * 100)}.csv"
        )

        threshold_table.to_csv(table_path)

        print(f"Saved table: {table_path}")

    # -----------------------------------------------------
    # Step 8. Save comparison plots
    # -----------------------------------------------------
    print("\n[Step 8] Save FPR95 plots")

    save_fpr95_plots(results_df, RESULT_DIR)
    save_method_comparison_plots(results_df, RESULT_DIR)

    print("\nExperiment completed.")


if __name__ == "__main__":
    main()

Using device: cuda
Loaded model from: ./results/cifar10_resnet18_clean.pth

[Step 1] Extract fixed train/test logits and features
Train logits shape     : (45000, 10)
Train features shape   : (45000, 512)
ID test logits shape   : (10000, 10)
ID test features shape : (10000, 512)
OOD test logits shape  : (26032, 10)
OOD test features shape: (26032, 512)

[Step 2] Fit KNN and ViM using clean ID training features
KNN k: 50
ViM residual start dimension: 256

[Step 3] Precompute fixed ID/OOD test scores
Computing ODIN ID/OOD test scores. This may take longer.

[Step 4] Run contaminated calibration experiments

Contamination ratio: 0.0%
Calibration ID samples : 5000
Calibration OOD samples: 0
Computing ODIN calibration scores. This may take longer.
MSP                          | calib=standard | trim=  0.0% | threshold=  0.774561 | AUROC= 90.77% | FPR95= 59.37%
MSP                          | calib=trimmed  | trim=  1.0% | threshold=  0.810495 | AUROC= 90.77% | FPR95= 55.02%
MSP              

# ID TRP Trade-off Analysis

In [ ]:
"""
FPR95-ID TPR Trade-off Analysis for Contaminated Calibration OOD Detection

Goal:
- Use a pretrained CIFAR-10 classifier.
- Keep the classifier fixed.
- Contaminate only the calibration set with SVHN samples.
- Apply standard calibration and robust lower-tail trimming calibration.
- Evaluate both:
    1. FPR95: how many OOD samples are incorrectly accepted as ID.
    2. ID TPR: how many ID samples are correctly accepted as ID.

Why this experiment is needed:
- In previous experiments, robust trimming reduced FPR95.
- However, aggressive trimming may also reject too many ID samples.
- Therefore, we need to analyze the trade-off between OOD rejection and ID preservation.

Dataset protocol:
- ID dataset        : CIFAR-10
- OOD dataset       : SVHN
- Train feature bank: CIFAR-10 train split only
- Clean calibration : CIFAR-10 calibration split only
- Contamination OOD : SVHN train split
- ID test           : CIFAR-10 test
- OOD test          : SVHN test

OOD methods:
1. MSP
2. Energy
3. KNN
4. ViM
5. ODIN

Calibration methods:
1. Standard calibration:
   - Use all calibration scores.
   - Threshold = 5th percentile of calibration scores.

2. Robust trimming calibration:
   - Remove the lowest q% calibration scores.
   - Then compute the 5th percentile threshold from the remaining scores.

Score convention:
- Higher score = more ID-like
- Lower score  = more OOD-like

Decision rule:
- score >= threshold -> ID
- score < threshold  -> OOD
"""

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset

from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"

MODEL_PATH = "./results/cifar10_resnet18_clean.pth"

BATCH_SIZE = 128
NUM_CLASSES = 10

# These values must match the clean baseline training split.
TRAIN_SIZE = 45000
CALIB_SIZE = 5000

# OOD method hyperparameters.
ENERGY_T = 1.0
KNN_K = 50
VIM_DIM = 256

# ODIN hyperparameters.
ODIN_T = 1000.0
ODIN_EPSILON = 0.0014

# Calibration contamination ratios.
CONTAMINATION_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20]

# Trimming ratios.
# 0.0 means standard calibration.
TRIM_RATIOS = [0.0, 0.01, 0.05, 0.10]

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

"""
The classifier was trained on CIFAR-10 normalized images.
Therefore, the same CIFAR-10 normalization is applied to both ID and OOD data.
"""

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# CIFAR-10 train data is used to reconstruct the same train/calibration split.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=test_transform
)

# Reconstruct the same split:
# - train_dataset: used for KNN feature bank and ViM statistics
# - clean_calib_dataset: used as the clean ID calibration pool
train_dataset, clean_calib_dataset = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

# CIFAR-10 test set is the ID test set.
id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

# SVHN train set is used as the OOD contamination source.
svhn_train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=test_transform
)

# SVHN test set is used as the final OOD test set.
ood_test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ood_test_loader = DataLoader(
    ood_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This wrapper exposes:
    - logits for MSP, Energy, and ODIN
    - penultimate features for KNN and ViM
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        # Modify the first convolution for 32x32 CIFAR-10 images.
        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # Remove the initial max pooling layer.
        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features before the final classifier.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True, return both logits and penultimate features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}\n"
        "Please run the clean baseline training script first."
    )

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 5. Contaminated calibration dataset
# =========================================================

def create_contaminated_calibration_dataset(
    clean_calib_dataset,
    svhn_train_dataset,
    contamination_ratio,
    total_calib_size=5000,
    seed=42
):
    """
    Create a contaminated calibration dataset.

    The total calibration size is fixed across contamination ratios.

    Example:
    - total_calib_size = 5000
    - contamination_ratio = 0.05
    - ID samples  = 4750
    - OOD samples = 250

    This ensures that threshold changes are caused by contamination,
    not by calibration set size changes.
    """
    rng = np.random.default_rng(seed)

    num_ood = int(total_calib_size * contamination_ratio)
    num_id = total_calib_size - num_ood

    id_indices = rng.choice(
        len(clean_calib_dataset),
        size=num_id,
        replace=False
    )

    id_subset = Subset(clean_calib_dataset, id_indices.tolist())

    if num_ood > 0:
        ood_indices = rng.choice(
            len(svhn_train_dataset),
            size=num_ood,
            replace=False
        )

        ood_subset = Subset(svhn_train_dataset, ood_indices.tolist())
        contaminated_dataset = ConcatDataset([id_subset, ood_subset])
    else:
        contaminated_dataset = id_subset

    return contaminated_dataset, num_id, num_ood


# =========================================================
# 6. Logit and feature extraction
# =========================================================

@torch.no_grad()
def extract_logits_and_features(model, data_loader):
    """
    Extract logits and penultimate features from a dataloader.

    Returns:
    - logits   : shape [N, num_classes]
    - features : shape [N, feature_dim]
    - labels   : shape [N]
    """
    model.eval()

    all_logits = []
    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(DEVICE)

        logits, features = model(images, return_features=True)

        all_logits.append(logits.detach().cpu())
        all_features.append(features.detach().cpu())
        all_labels.append(torch.as_tensor(labels))

    logits = torch.cat(all_logits, dim=0).numpy()
    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    return logits, features, labels


# =========================================================
# 7. OOD score functions
# =========================================================

def compute_msp_from_logits(logits):
    """
    Compute Maximum Softmax Probability.

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    probs = torch.softmax(logits_tensor, dim=1)
    scores = probs.max(dim=1).values.numpy()
    return scores


def compute_energy_from_logits(logits, temperature=1.0):
    """
    Compute Energy-based ID scores.

    Standard energy:
        E(x) = -T * logsumexp(logits / T)

    This function returns -E(x):
        score(x) = T * logsumexp(logits / T)

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    scores = temperature * torch.logsumexp(
        logits_tensor / temperature,
        dim=1
    ).numpy()
    return scores


def fit_knn_feature_bank(train_features, k=50):
    """
    Fit KNN on clean ID training features.
    """
    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=-1
    )
    knn.fit(train_features)
    return knn


def compute_knn_scores(knn, features, k=50):
    """
    Compute KNN-based ID scores.

    Score = negative distance to the k-th nearest ID training feature.
    Higher score means more ID-like.
    """
    distances, _ = knn.kneighbors(features, n_neighbors=k)
    kth_distance = distances[:, -1]
    scores = -kth_distance
    return scores


def fit_vim(train_features, train_logits, vim_dim=256):
    """
    Fit simplified ViM statistics using clean ID training data.

    Simplified ViM steps:
    1. Estimate ID feature mean.
    2. Compute principal directions using SVD.
    3. Define residual subspace as the complement of the principal subspace.
    4. Scale residual norm using alpha.
    """
    feature_mean = np.mean(train_features, axis=0, keepdims=True)

    centered_features = train_features - feature_mean

    _, _, vt = np.linalg.svd(centered_features, full_matrices=False)

    feature_dim = train_features.shape[1]
    vim_dim = min(vim_dim, feature_dim - 1)

    residual_space = vt[vim_dim:].T

    train_residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    train_max_logit = np.max(train_logits, axis=1)

    alpha = np.mean(train_max_logit) / (np.mean(train_residual_norm) + 1e-12)

    vim_stats = {
        "feature_mean": feature_mean,
        "residual_space": residual_space,
        "alpha": alpha,
        "vim_dim": vim_dim
    }

    return vim_stats


def compute_vim_scores(features, logits, vim_stats):
    """
    Compute simplified ViM ID scores.

    virtual_logit = alpha * residual_norm
    score = max_original_logit - virtual_logit

    Higher score means more ID-like.
    """
    feature_mean = vim_stats["feature_mean"]
    residual_space = vim_stats["residual_space"]
    alpha = vim_stats["alpha"]

    centered_features = features - feature_mean

    residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    virtual_logit = alpha * residual_norm
    max_original_logit = np.max(logits, axis=1)

    scores = max_original_logit - virtual_logit

    return scores


def compute_odin_scores(model, data_loader, temperature=1000.0, epsilon=0.0014):
    """
    Compute ODIN scores.

    ODIN uses:
    1. Temperature scaling
    2. Small input perturbation based on gradients

    The returned score is the maximum softmax probability after ODIN processing.
    Higher score means more ID-like.
    """
    model.eval()

    all_scores = []

    for images, _ in data_loader:
        images = images.to(DEVICE)
        images.requires_grad_(True)

        logits = model(images)
        scaled_logits = logits / temperature

        # Use model predictions as pseudo-labels.
        pseudo_labels = scaled_logits.argmax(dim=1)

        loss = F.cross_entropy(scaled_logits, pseudo_labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = torch.sign(images.grad.data)

        perturbed_images = images - epsilon * gradient_sign
        perturbed_images = perturbed_images.detach()

        with torch.no_grad():
            perturbed_logits = model(perturbed_images)
            perturbed_scaled_logits = perturbed_logits / temperature
            probs = torch.softmax(perturbed_scaled_logits, dim=1)

            scores = probs.max(dim=1).values

        all_scores.append(scores.detach().cpu())

    scores = torch.cat(all_scores, dim=0).numpy()

    return scores


# =========================================================
# 8. Threshold and metric functions
# =========================================================

def get_trimmed_threshold_at_95_tpr(calib_scores, trim_ratio=0.0):
    """
    Estimate a threshold using lower-tail trimming.

    Procedure:
    1. Sort calibration scores in ascending order.
    2. Remove the lowest trim_ratio portion.
    3. Compute the 5th percentile threshold from the remaining scores.

    trim_ratio = 0.0 is equivalent to standard calibration.
    """
    calib_scores = np.asarray(calib_scores)

    if trim_ratio < 0.0 or trim_ratio >= 1.0:
        raise ValueError("trim_ratio must be in [0.0, 1.0).")

    sorted_scores = np.sort(calib_scores)

    num_samples = len(sorted_scores)
    num_trim = int(num_samples * trim_ratio)

    trimmed_scores = sorted_scores[num_trim:]

    if len(trimmed_scores) == 0:
        raise ValueError("All calibration samples were trimmed.")

    threshold = np.percentile(trimmed_scores, 5)

    return threshold


def compute_fpr95(ood_scores, threshold):
    """
    Compute FPR95.

    FPR95 is the fraction of OOD samples incorrectly accepted as ID.
    """
    false_positive = np.sum(ood_scores >= threshold)
    fpr95 = false_positive / len(ood_scores)
    return fpr95


def compute_id_tpr(id_scores, threshold):
    """
    Compute ID TPR.

    ID TPR is the fraction of ID samples correctly accepted as ID.
    """
    id_tpr = np.mean(id_scores >= threshold)
    return id_tpr


def compute_id_rejection_rate(id_scores, threshold):
    """
    Compute ID rejection rate.

    ID rejection rate is the fraction of ID samples incorrectly rejected as OOD.
    """
    id_rejection = np.mean(id_scores < threshold)
    return id_rejection


def compute_auroc(id_scores, ood_scores):
    """
    Compute AUROC.

    Label convention:
    - ID  = 1
    - OOD = 0

    Score convention:
    - Higher score means more ID-like.
    """
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores))
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores
    ])

    auroc = roc_auc_score(y_true, y_score)

    return auroc


def evaluate_ood_method(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores,
    trim_ratio
):
    """
    Evaluate one OOD method under one calibration setting.

    Metrics:
    - AUROC
    - FPR95
    - ID TPR
    - ID rejection rate

    trim_ratio:
    - 0.0 means standard calibration.
    - >0.0 means robust lower-tail trimming calibration.
    """
    threshold = get_trimmed_threshold_at_95_tpr(
        calib_scores=calib_scores,
        trim_ratio=trim_ratio
    )

    auroc = compute_auroc(
        id_scores=id_test_scores,
        ood_scores=ood_test_scores
    )

    fpr95 = compute_fpr95(
        ood_scores=ood_test_scores,
        threshold=threshold
    )

    id_tpr = compute_id_tpr(
        id_scores=id_test_scores,
        threshold=threshold
    )

    id_rejection = compute_id_rejection_rate(
        id_scores=id_test_scores,
        threshold=threshold
    )

    calibration_type = "standard" if trim_ratio == 0.0 else "trimmed"

    result = {
        "method": method_name,
        "calibration_type": calibration_type,
        "trim_ratio": float(trim_ratio),
        "trim_percent": float(trim_ratio * 100),
        "threshold": float(threshold),

        "auroc": float(auroc),
        "auroc_percent": float(auroc * 100),

        "fpr95": float(fpr95),
        "fpr95_percent": float(fpr95 * 100),

        "id_tpr": float(id_tpr),
        "id_tpr_percent": float(id_tpr * 100),

        "id_rejection_rate": float(id_rejection),
        "id_rejection_percent": float(id_rejection * 100)
    }

    return result


# =========================================================
# 9. Plotting functions
# =========================================================

def save_tradeoff_plots(results_df, result_dir):
    """
    Save FPR95-ID TPR trade-off plots for each OOD method.

    Each plot:
    - x-axis: ID TPR
    - y-axis: FPR95
    - one point per contamination ratio and trim ratio
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for trim_ratio in sorted(subset["trim_ratio"].unique()):
            trim_df = subset[subset["trim_ratio"] == trim_ratio].sort_values(
                by="contamination_percent"
            )

            label = "standard" if trim_ratio == 0.0 else f"trim {trim_ratio * 100:.1f}%"

            plt.plot(
                trim_df["id_tpr_percent"],
                trim_df["fpr95_percent"],
                marker="o",
                label=label
            )

        plt.xlabel("ID TPR (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"FPR95-ID TPR trade-off | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_method_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(
            result_dir,
            f"tradeoff_fpr95_id_tpr_{safe_method_name}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved trade-off plot: {save_path}")


def save_fpr95_vs_contamination_plots(results_df, result_dir):
    """
    Save FPR95 vs contamination ratio plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for trim_ratio in sorted(subset["trim_ratio"].unique()):
            trim_df = subset[subset["trim_ratio"] == trim_ratio].sort_values(
                by="contamination_percent"
            )

            label = "standard" if trim_ratio == 0.0 else f"trim {trim_ratio * 100:.1f}%"

            plt.plot(
                trim_df["contamination_percent"],
                trim_df["fpr95_percent"],
                marker="o",
                label=label
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"FPR95 under calibration contamination | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_method_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(
            result_dir,
            f"fpr95_vs_contamination_{safe_method_name}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved FPR95 plot: {save_path}")


def save_id_tpr_vs_contamination_plots(results_df, result_dir):
    """
    Save ID TPR vs contamination ratio plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for trim_ratio in sorted(subset["trim_ratio"].unique()):
            trim_df = subset[subset["trim_ratio"] == trim_ratio].sort_values(
                by="contamination_percent"
            )

            label = "standard" if trim_ratio == 0.0 else f"trim {trim_ratio * 100:.1f}%"

            plt.plot(
                trim_df["contamination_percent"],
                trim_df["id_tpr_percent"],
                marker="o",
                label=label
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("ID TPR (%)")
        plt.title(f"ID TPR under calibration contamination | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_method_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(
            result_dir,
            f"id_tpr_vs_contamination_{safe_method_name}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved ID TPR plot: {save_path}")


# =========================================================
# 10. Main experiment
# =========================================================

def main():
    """
    Run the full FPR95-ID TPR trade-off experiment.

    Main pipeline:
    1. Extract fixed train/test logits and features.
    2. Fit KNN and ViM using clean ID training data.
    3. Precompute fixed ID/OOD test scores for all methods.
    4. For each contamination ratio:
        a. Create contaminated calibration set.
        b. Compute calibration scores.
        c. Estimate thresholds with different trim ratios.
        d. Compute AUROC, FPR95, ID TPR, and ID rejection.
    5. Save result tables and plots.
    """

    # -----------------------------------------------------
    # Step 1. Extract fixed train/test logits and features
    # -----------------------------------------------------
    print("\n[Step 1] Extract fixed train/test logits and features")

    train_logits, train_features, _ = extract_logits_and_features(
        model,
        train_loader
    )

    id_test_logits, id_test_features, _ = extract_logits_and_features(
        model,
        id_test_loader
    )

    ood_test_logits, ood_test_features, _ = extract_logits_and_features(
        model,
        ood_test_loader
    )

    print(f"Train logits shape      : {train_logits.shape}")
    print(f"Train features shape    : {train_features.shape}")
    print(f"ID test logits shape    : {id_test_logits.shape}")
    print(f"ID test features shape  : {id_test_features.shape}")
    print(f"OOD test logits shape   : {ood_test_logits.shape}")
    print(f"OOD test features shape : {ood_test_features.shape}")

    # -----------------------------------------------------
    # Step 2. Fit KNN and ViM
    # -----------------------------------------------------
    print("\n[Step 2] Fit KNN and ViM using clean ID training features")

    knn = fit_knn_feature_bank(
        train_features=train_features,
        k=KNN_K
    )

    vim_stats = fit_vim(
        train_features=train_features,
        train_logits=train_logits,
        vim_dim=VIM_DIM
    )

    print(f"KNN k: {KNN_K}")
    print(f"ViM residual start dimension: {vim_stats['vim_dim']}")

    # -----------------------------------------------------
    # Step 3. Precompute fixed ID/OOD test scores
    # -----------------------------------------------------
    print("\n[Step 3] Precompute fixed ID/OOD test scores")

    fixed_test_scores = {}

    fixed_test_scores["MSP"] = {
        "id": compute_msp_from_logits(id_test_logits),
        "ood": compute_msp_from_logits(ood_test_logits)
    }

    fixed_test_scores["Energy"] = {
        "id": compute_energy_from_logits(
            id_test_logits,
            temperature=ENERGY_T
        ),
        "ood": compute_energy_from_logits(
            ood_test_logits,
            temperature=ENERGY_T
        )
    }

    fixed_test_scores[f"KNN-k{KNN_K}"] = {
        "id": compute_knn_scores(
            knn=knn,
            features=id_test_features,
            k=KNN_K
        ),
        "ood": compute_knn_scores(
            knn=knn,
            features=ood_test_features,
            k=KNN_K
        )
    }

    fixed_test_scores[f"ViM-dim{vim_stats['vim_dim']}"] = {
        "id": compute_vim_scores(
            features=id_test_features,
            logits=id_test_logits,
            vim_stats=vim_stats
        ),
        "ood": compute_vim_scores(
            features=ood_test_features,
            logits=ood_test_logits,
            vim_stats=vim_stats
        )
    }

    print("Computing ODIN ID/OOD test scores. This may take longer.")

    fixed_test_scores[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = {
        "id": compute_odin_scores(
            model=model,
            data_loader=id_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        ),
        "ood": compute_odin_scores(
            model=model,
            data_loader=ood_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )
    }

    # -----------------------------------------------------
    # Step 4. Run experiments for each contamination ratio
    # -----------------------------------------------------
    print("\n[Step 4] Run contaminated calibration experiments")

    all_results = []

    for contamination_ratio in CONTAMINATION_RATIOS:
        print("\n" + "=" * 110)
        print(f"Contamination ratio: {contamination_ratio * 100:.1f}%")
        print("=" * 110)

        contaminated_calib_dataset, num_id, num_ood = create_contaminated_calibration_dataset(
            clean_calib_dataset=clean_calib_dataset,
            svhn_train_dataset=svhn_train_dataset,
            contamination_ratio=contamination_ratio,
            total_calib_size=CALIB_SIZE,
            seed=42 + int(contamination_ratio * 10000)
        )

        contaminated_calib_loader = DataLoader(
            contaminated_calib_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        print(f"Calibration ID samples : {num_id}")
        print(f"Calibration OOD samples: {num_ood}")

        calib_logits, calib_features, _ = extract_logits_and_features(
            model,
            contaminated_calib_loader
        )

        calib_scores_dict = {}

        calib_scores_dict["MSP"] = compute_msp_from_logits(calib_logits)

        calib_scores_dict["Energy"] = compute_energy_from_logits(
            calib_logits,
            temperature=ENERGY_T
        )

        calib_scores_dict[f"KNN-k{KNN_K}"] = compute_knn_scores(
            knn=knn,
            features=calib_features,
            k=KNN_K
        )

        calib_scores_dict[f"ViM-dim{vim_stats['vim_dim']}"] = compute_vim_scores(
            features=calib_features,
            logits=calib_logits,
            vim_stats=vim_stats
        )

        print("Computing ODIN calibration scores. This may take longer.")

        calib_scores_dict[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = compute_odin_scores(
            model=model,
            data_loader=contaminated_calib_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )

        for method_name, calib_scores in calib_scores_dict.items():
            for trim_ratio in TRIM_RATIOS:
                result = evaluate_ood_method(
                    method_name=method_name,
                    calib_scores=calib_scores,
                    id_test_scores=fixed_test_scores[method_name]["id"],
                    ood_test_scores=fixed_test_scores[method_name]["ood"],
                    trim_ratio=trim_ratio
                )

                row = {
                    "contamination_ratio": float(contamination_ratio),
                    "contamination_percent": float(contamination_ratio * 100),
                    "num_calib_id": int(num_id),
                    "num_calib_ood": int(num_ood),
                    **result
                }

                all_results.append(row)

                print(
                    f"{method_name:<28} | "
                    f"calib={result['calibration_type']:<8} | "
                    f"trim={trim_ratio * 100:>5.1f}% | "
                    f"threshold={result['threshold']:>10.6f} | "
                    f"AUROC={result['auroc_percent']:>6.2f}% | "
                    f"FPR95={result['fpr95_percent']:>6.2f}% | "
                    f"ID_TPR={result['id_tpr_percent']:>6.2f}% | "
                    f"ID_Reject={result['id_rejection_percent']:>6.2f}%"
                )

    # -----------------------------------------------------
    # Step 5. Save full results
    # -----------------------------------------------------
    results_df = pd.DataFrame(all_results)

    full_csv_path = os.path.join(
        RESULT_DIR,
        "tradeoff_full_results.csv"
    )

    full_npy_path = os.path.join(
        RESULT_DIR,
        "tradeoff_full_results.npy"
    )

    results_df.to_csv(full_csv_path, index=False)
    np.save(full_npy_path, all_results, allow_pickle=True)

    print("\nSaved full trade-off results to:")
    print(full_csv_path)
    print(full_npy_path)

    # -----------------------------------------------------
    # Step 6. Save FPR95 tables by trim ratio
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("FPR95 (%) tables by trim ratio")
    print("=" * 100)

    for trim_ratio in TRIM_RATIOS:
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        fpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="fpr95_percent"
        )

        print("\n" + "-" * 100)
        print(f"Trim ratio: {trim_ratio * 100:.1f}%")
        print("-" * 100)
        print(fpr_table.round(2))

        table_path = os.path.join(
            RESULT_DIR,
            f"tradeoff_fpr95_table_trim_{int(trim_ratio * 100)}.csv"
        )

        fpr_table.to_csv(table_path)
        print(f"Saved table: {table_path}")

    # -----------------------------------------------------
    # Step 7. Save ID TPR tables by trim ratio
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("ID TPR (%) tables by trim ratio")
    print("=" * 100)

    for trim_ratio in TRIM_RATIOS:
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        id_tpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="id_tpr_percent"
        )

        print("\n" + "-" * 100)
        print(f"Trim ratio: {trim_ratio * 100:.1f}%")
        print("-" * 100)
        print(id_tpr_table.round(2))

        table_path = os.path.join(
            RESULT_DIR,
            f"tradeoff_id_tpr_table_trim_{int(trim_ratio * 100)}.csv"
        )

        id_tpr_table.to_csv(table_path)
        print(f"Saved table: {table_path}")

    # -----------------------------------------------------
    # Step 8. Save ID rejection tables by trim ratio
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("ID rejection (%) tables by trim ratio")
    print("=" * 100)

    for trim_ratio in TRIM_RATIOS:
        subset = results_df[results_df["trim_ratio"] == trim_ratio]

        id_rejection_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="id_rejection_percent"
        )

        print("\n" + "-" * 100)
        print(f"Trim ratio: {trim_ratio * 100:.1f}%")
        print("-" * 100)
        print(id_rejection_table.round(2))

        table_path = os.path.join(
            RESULT_DIR,
            f"tradeoff_id_rejection_table_trim_{int(trim_ratio * 100)}.csv"
        )

        id_rejection_table.to_csv(table_path)
        print(f"Saved table: {table_path}")

    # -----------------------------------------------------
    # Step 9. Save compact trade-off table
    # -----------------------------------------------------
    tradeoff_columns = [
        "method",
        "contamination_percent",
        "trim_percent",
        "threshold",
        "auroc_percent",
        "fpr95_percent",
        "id_tpr_percent",
        "id_rejection_percent"
    ]

    tradeoff_df = results_df[tradeoff_columns].copy()

    tradeoff_csv_path = os.path.join(
        RESULT_DIR,
        "fpr95_id_tpr_tradeoff_results.csv"
    )

    tradeoff_df.to_csv(tradeoff_csv_path, index=False)

    print("\nSaved compact FPR95-ID TPR trade-off table to:")
    print(tradeoff_csv_path)

    # -----------------------------------------------------
    # Step 10. Save plots
    # -----------------------------------------------------
    print("\n[Step 10] Save plots")

    save_tradeoff_plots(results_df, RESULT_DIR)
    save_fpr95_vs_contamination_plots(results_df, RESULT_DIR)
    save_id_tpr_vs_contamination_plots(results_df, RESULT_DIR)

    print("\nExperiment completed.")


if __name__ == "__main__":
    main()

Using device: cuda
Loaded model from: ./results/cifar10_resnet18_clean.pth

[Step 1] Extract fixed train/test logits and features
Train logits shape      : (45000, 10)
Train features shape    : (45000, 512)
ID test logits shape    : (10000, 10)
ID test features shape  : (10000, 512)
OOD test logits shape   : (26032, 10)
OOD test features shape : (26032, 512)

[Step 2] Fit KNN and ViM using clean ID training features
KNN k: 50
ViM residual start dimension: 256

[Step 3] Precompute fixed ID/OOD test scores
Computing ODIN ID/OOD test scores. This may take longer.

[Step 4] Run contaminated calibration experiments

Contamination ratio: 0.0%
Calibration ID samples : 5000
Calibration OOD samples: 0
Computing ODIN calibration scores. This may take longer.
MSP                          | calib=standard | trim=  0.0% | threshold=  0.774561 | AUROC= 90.77% | FPR95= 59.37% | ID_TPR= 94.72% | ID_Reject=  5.28%
MSP                          | calib=trimmed  | trim=  1.0% | threshold=  0.810495 | AURO

# Adaptive Trimming - IQR

In [ ]:
"""
IQR-based Adaptive Trimming Calibration for Post-hoc OOD Detection

Goal:
- Use a pretrained CIFAR-10 classifier.
- Contaminate only the calibration set with SVHN samples.
- Compare standard calibration with IQR-based adaptive trimming.
- Evaluate FPR95, ID TPR, and ID rejection rate.

Dataset protocol:
- ID dataset        : CIFAR-10
- OOD dataset       : SVHN
- Train feature bank: CIFAR-10 train split only
- Clean calibration : CIFAR-10 calibration split only
- Contamination OOD : SVHN train split
- ID test           : CIFAR-10 test
- OOD test          : SVHN test

OOD methods:
1. MSP
2. Energy
3. KNN
4. ViM
5. ODIN

Calibration methods:
1. Standard calibration:
   - Use all calibration scores.
   - Threshold = 5th percentile of calibration scores.

2. IQR adaptive trimming:
   - Compute Q1, Q3, and IQR from calibration scores.
   - lower_bound = Q1 - lambda * IQR
   - Remove scores below lower_bound.
   - Compute the 5th percentile threshold from the remaining scores.

Score convention:
- Higher score = more ID-like
- Lower score  = more OOD-like

Decision rule:
- score >= threshold -> ID
- score < threshold  -> OOD
"""

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset

from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"

MODEL_PATH = "./results/cifar10_resnet18_clean.pth"

BATCH_SIZE = 128
NUM_CLASSES = 10

# These values must match the clean baseline training split.
TRAIN_SIZE = 45000
CALIB_SIZE = 5000

# OOD method hyperparameters.
ENERGY_T = 1.0
KNN_K = 50
VIM_DIM = 256

# ODIN hyperparameters.
ODIN_T = 1000.0
ODIN_EPSILON = 0.0014

# Calibration contamination ratios.
CONTAMINATION_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20]

# IQR lambda candidates.
# Smaller lambda removes more low-score samples.
# Larger lambda is more conservative.
IQR_LAMBDAS = [0.5, 1.0, 1.5]

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# CIFAR-10 train data is used to reconstruct the same train/calibration split.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=test_transform
)

train_dataset, clean_calib_dataset = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

svhn_train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=test_transform
)

ood_test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ood_test_loader = DataLoader(
    ood_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This wrapper exposes:
    - logits for MSP, Energy, and ODIN
    - penultimate features for KNN and ViM
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features before the classifier head.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True, return both logits and features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}\n"
        "Please run the clean baseline training script first."
    )

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 5. Contaminated calibration dataset
# =========================================================

def create_contaminated_calibration_dataset(
    clean_calib_dataset,
    svhn_train_dataset,
    contamination_ratio,
    total_calib_size=5000,
    seed=42
):
    """
    Create a contaminated calibration dataset.

    The total calibration size is fixed.

    Example:
    - contamination_ratio = 0.05
    - total_calib_size = 5000
    - ID samples  = 4750
    - OOD samples = 250
    """
    rng = np.random.default_rng(seed)

    num_ood = int(total_calib_size * contamination_ratio)
    num_id = total_calib_size - num_ood

    id_indices = rng.choice(
        len(clean_calib_dataset),
        size=num_id,
        replace=False
    )

    id_subset = Subset(clean_calib_dataset, id_indices.tolist())

    if num_ood > 0:
        ood_indices = rng.choice(
            len(svhn_train_dataset),
            size=num_ood,
            replace=False
        )

        ood_subset = Subset(svhn_train_dataset, ood_indices.tolist())
        contaminated_dataset = ConcatDataset([id_subset, ood_subset])
    else:
        contaminated_dataset = id_subset

    return contaminated_dataset, num_id, num_ood


# =========================================================
# 6. Logit and feature extraction
# =========================================================

@torch.no_grad()
def extract_logits_and_features(model, data_loader):
    """
    Extract logits and penultimate features from a dataloader.
    """
    model.eval()

    all_logits = []
    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(DEVICE)

        logits, features = model(images, return_features=True)

        all_logits.append(logits.detach().cpu())
        all_features.append(features.detach().cpu())
        all_labels.append(torch.as_tensor(labels))

    logits = torch.cat(all_logits, dim=0).numpy()
    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    return logits, features, labels


# =========================================================
# 7. OOD score functions
# =========================================================

def compute_msp_from_logits(logits):
    """
    Compute Maximum Softmax Probability.

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    probs = torch.softmax(logits_tensor, dim=1)
    scores = probs.max(dim=1).values.numpy()
    return scores


def compute_energy_from_logits(logits, temperature=1.0):
    """
    Compute Energy-based ID scores.

    This returns -E(x):
        score = T * logsumexp(logits / T)

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    scores = temperature * torch.logsumexp(
        logits_tensor / temperature,
        dim=1
    ).numpy()
    return scores


def fit_knn_feature_bank(train_features, k=50):
    """
    Fit KNN on clean ID training features.
    """
    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=-1
    )
    knn.fit(train_features)
    return knn


def compute_knn_scores(knn, features, k=50):
    """
    Compute KNN-based ID scores.

    Score = negative distance to the k-th nearest ID training feature.
    Higher score means more ID-like.
    """
    distances, _ = knn.kneighbors(features, n_neighbors=k)
    kth_distance = distances[:, -1]
    scores = -kth_distance
    return scores


def fit_vim(train_features, train_logits, vim_dim=256):
    """
    Fit simplified ViM statistics using clean ID training data.
    """
    feature_mean = np.mean(train_features, axis=0, keepdims=True)

    centered_features = train_features - feature_mean

    _, _, vt = np.linalg.svd(centered_features, full_matrices=False)

    feature_dim = train_features.shape[1]
    vim_dim = min(vim_dim, feature_dim - 1)

    residual_space = vt[vim_dim:].T

    train_residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    train_max_logit = np.max(train_logits, axis=1)

    alpha = np.mean(train_max_logit) / (np.mean(train_residual_norm) + 1e-12)

    vim_stats = {
        "feature_mean": feature_mean,
        "residual_space": residual_space,
        "alpha": alpha,
        "vim_dim": vim_dim
    }

    return vim_stats


def compute_vim_scores(features, logits, vim_stats):
    """
    Compute simplified ViM ID scores.

    score = max_original_logit - alpha * residual_norm

    Higher score means more ID-like.
    """
    feature_mean = vim_stats["feature_mean"]
    residual_space = vim_stats["residual_space"]
    alpha = vim_stats["alpha"]

    centered_features = features - feature_mean

    residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    virtual_logit = alpha * residual_norm
    max_original_logit = np.max(logits, axis=1)

    scores = max_original_logit - virtual_logit
    return scores


def compute_odin_scores(model, data_loader, temperature=1000.0, epsilon=0.0014):
    """
    Compute ODIN scores.

    Higher score means more ID-like.
    """
    model.eval()

    all_scores = []

    for images, _ in data_loader:
        images = images.to(DEVICE)
        images.requires_grad_(True)

        logits = model(images)
        scaled_logits = logits / temperature

        pseudo_labels = scaled_logits.argmax(dim=1)
        loss = F.cross_entropy(scaled_logits, pseudo_labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = torch.sign(images.grad.data)

        perturbed_images = images - epsilon * gradient_sign
        perturbed_images = perturbed_images.detach()

        with torch.no_grad():
            perturbed_logits = model(perturbed_images)
            perturbed_scaled_logits = perturbed_logits / temperature
            probs = torch.softmax(perturbed_scaled_logits, dim=1)

            scores = probs.max(dim=1).values

        all_scores.append(scores.detach().cpu())

    scores = torch.cat(all_scores, dim=0).numpy()
    return scores


# =========================================================
# 8. Threshold and metric functions
# =========================================================

def get_standard_threshold(calib_scores):
    """
    Standard calibration threshold.

    Threshold = 5th percentile of all calibration scores.
    """
    return np.percentile(calib_scores, 5)


def get_iqr_adaptive_threshold(calib_scores, lambda_iqr=1.5):
    """
    IQR-based adaptive trimming threshold.

    Procedure:
    1. Compute Q1, Q3, and IQR.
    2. Define lower_bound = Q1 - lambda_iqr * IQR.
    3. Remove scores below lower_bound.
    4. Compute threshold as the 5th percentile of the remaining scores.

    Since lower scores are more OOD-like, only lower-tail samples are removed.
    """
    calib_scores = np.asarray(calib_scores)

    q1 = np.percentile(calib_scores, 25)
    q3 = np.percentile(calib_scores, 75)
    iqr = q3 - q1

    lower_bound = q1 - lambda_iqr * iqr

    keep_mask = calib_scores >= lower_bound
    kept_scores = calib_scores[keep_mask]
    removed_scores = calib_scores[~keep_mask]

    # Safety fallback:
    # If no samples are removed or too few samples remain, this still works.
    # If all samples are removed, fall back to standard calibration.
    if len(kept_scores) == 0:
        kept_scores = calib_scores
        keep_mask = np.ones_like(calib_scores, dtype=bool)
        removed_scores = np.array([])

    threshold = np.percentile(kept_scores, 5)

    removed_ratio = len(removed_scores) / len(calib_scores)

    stats = {
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower_bound": float(lower_bound),
        "num_removed": int(len(removed_scores)),
        "removed_ratio": float(removed_ratio),
        "removed_percent": float(removed_ratio * 100),
        "num_kept": int(len(kept_scores))
    }

    return threshold, stats


def compute_fpr95(ood_scores, threshold):
    """
    Compute FPR95.

    FPR95 is the fraction of OOD samples incorrectly accepted as ID.
    """
    return np.mean(ood_scores >= threshold)


def compute_id_tpr(id_scores, threshold):
    """
    Compute ID TPR.

    ID TPR is the fraction of ID samples correctly accepted as ID.
    """
    return np.mean(id_scores >= threshold)


def compute_id_rejection_rate(id_scores, threshold):
    """
    Compute ID rejection rate.

    ID rejection rate is the fraction of ID samples incorrectly rejected as OOD.
    """
    return np.mean(id_scores < threshold)


def compute_auroc(id_scores, ood_scores):
    """
    Compute AUROC.

    ID label  = 1
    OOD label = 0
    """
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores))
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores
    ])

    return roc_auc_score(y_true, y_score)


def evaluate_standard(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores
):
    """
    Evaluate standard calibration.
    """
    threshold = get_standard_threshold(calib_scores)

    auroc = compute_auroc(id_test_scores, ood_test_scores)
    fpr95 = compute_fpr95(ood_test_scores, threshold)
    id_tpr = compute_id_tpr(id_test_scores, threshold)
    id_reject = compute_id_rejection_rate(id_test_scores, threshold)

    return {
        "method": method_name,
        "calibration_type": "standard",
        "lambda_iqr": np.nan,
        "threshold": float(threshold),

        "auroc": float(auroc),
        "auroc_percent": float(auroc * 100),

        "fpr95": float(fpr95),
        "fpr95_percent": float(fpr95 * 100),

        "id_tpr": float(id_tpr),
        "id_tpr_percent": float(id_tpr * 100),

        "id_rejection_rate": float(id_reject),
        "id_rejection_percent": float(id_reject * 100),

        "q1": np.nan,
        "q3": np.nan,
        "iqr": np.nan,
        "lower_bound": np.nan,
        "num_removed": 0,
        "removed_ratio": 0.0,
        "removed_percent": 0.0,
        "num_kept": len(calib_scores)
    }


def evaluate_iqr_adaptive(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores,
    lambda_iqr
):
    """
    Evaluate IQR-based adaptive trimming calibration.
    """
    threshold, stats = get_iqr_adaptive_threshold(
        calib_scores=calib_scores,
        lambda_iqr=lambda_iqr
    )

    auroc = compute_auroc(id_test_scores, ood_test_scores)
    fpr95 = compute_fpr95(ood_test_scores, threshold)
    id_tpr = compute_id_tpr(id_test_scores, threshold)
    id_reject = compute_id_rejection_rate(id_test_scores, threshold)

    return {
        "method": method_name,
        "calibration_type": "iqr_adaptive",
        "lambda_iqr": float(lambda_iqr),
        "threshold": float(threshold),

        "auroc": float(auroc),
        "auroc_percent": float(auroc * 100),

        "fpr95": float(fpr95),
        "fpr95_percent": float(fpr95 * 100),

        "id_tpr": float(id_tpr),
        "id_tpr_percent": float(id_tpr * 100),

        "id_rejection_rate": float(id_reject),
        "id_rejection_percent": float(id_reject * 100),

        **stats
    }


# =========================================================
# 9. Plotting functions
# =========================================================

def save_iqr_fpr95_plots(results_df, result_dir):
    """
    Save FPR95 vs contamination plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        standard_df = subset[subset["calibration_type"] == "standard"].sort_values(
            by="contamination_percent"
        )

        plt.plot(
            standard_df["contamination_percent"],
            standard_df["fpr95_percent"],
            marker="o",
            label="standard"
        )

        for lambda_iqr in sorted(subset["lambda_iqr"].dropna().unique()):
            lambda_df = subset[
                (subset["calibration_type"] == "iqr_adaptive") &
                (subset["lambda_iqr"] == lambda_iqr)
            ].sort_values(by="contamination_percent")

            plt.plot(
                lambda_df["contamination_percent"],
                lambda_df["fpr95_percent"],
                marker="o",
                label=f"IQR λ={lambda_iqr}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"IQR adaptive calibration: FPR95 | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"iqr_fpr95_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_iqr_id_tpr_plots(results_df, result_dir):
    """
    Save ID TPR vs contamination plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        standard_df = subset[subset["calibration_type"] == "standard"].sort_values(
            by="contamination_percent"
        )

        plt.plot(
            standard_df["contamination_percent"],
            standard_df["id_tpr_percent"],
            marker="o",
            label="standard"
        )

        for lambda_iqr in sorted(subset["lambda_iqr"].dropna().unique()):
            lambda_df = subset[
                (subset["calibration_type"] == "iqr_adaptive") &
                (subset["lambda_iqr"] == lambda_iqr)
            ].sort_values(by="contamination_percent")

            plt.plot(
                lambda_df["contamination_percent"],
                lambda_df["id_tpr_percent"],
                marker="o",
                label=f"IQR λ={lambda_iqr}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("ID TPR (%)")
        plt.title(f"IQR adaptive calibration: ID TPR | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"iqr_id_tpr_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_iqr_removed_ratio_plots(results_df, result_dir):
    """
    Save removed ratio vs contamination plots for each OOD method.
    """
    adaptive_df = results_df[results_df["calibration_type"] == "iqr_adaptive"]

    for method in sorted(adaptive_df["method"].unique()):
        subset = adaptive_df[adaptive_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for lambda_iqr in sorted(subset["lambda_iqr"].unique()):
            lambda_df = subset[subset["lambda_iqr"] == lambda_iqr].sort_values(
                by="contamination_percent"
            )

            plt.plot(
                lambda_df["contamination_percent"],
                lambda_df["removed_percent"],
                marker="o",
                label=f"IQR λ={lambda_iqr}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("Removed calibration samples (%)")
        plt.title(f"IQR adaptive trimming ratio | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"iqr_removed_ratio_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


# =========================================================
# 10. Main experiment
# =========================================================

def main():
    """
    Run the full IQR adaptive trimming experiment.
    """

    # -----------------------------------------------------
    # Step 1. Extract fixed train/test logits and features
    # -----------------------------------------------------
    print("\n[Step 1] Extract fixed train/test logits and features")

    train_logits, train_features, _ = extract_logits_and_features(
        model,
        train_loader
    )

    id_test_logits, id_test_features, _ = extract_logits_and_features(
        model,
        id_test_loader
    )

    ood_test_logits, ood_test_features, _ = extract_logits_and_features(
        model,
        ood_test_loader
    )

    print(f"Train features shape   : {train_features.shape}")
    print(f"ID test features shape : {id_test_features.shape}")
    print(f"OOD test features shape: {ood_test_features.shape}")

    # -----------------------------------------------------
    # Step 2. Fit KNN and ViM
    # -----------------------------------------------------
    print("\n[Step 2] Fit KNN and ViM")

    knn = fit_knn_feature_bank(
        train_features=train_features,
        k=KNN_K
    )

    vim_stats = fit_vim(
        train_features=train_features,
        train_logits=train_logits,
        vim_dim=VIM_DIM
    )

    print(f"KNN k: {KNN_K}")
    print(f"ViM dim: {vim_stats['vim_dim']}")

    # -----------------------------------------------------
    # Step 3. Precompute fixed ID/OOD test scores
    # -----------------------------------------------------
    print("\n[Step 3] Precompute fixed ID/OOD test scores")

    fixed_test_scores = {}

    fixed_test_scores["MSP"] = {
        "id": compute_msp_from_logits(id_test_logits),
        "ood": compute_msp_from_logits(ood_test_logits)
    }

    fixed_test_scores["Energy"] = {
        "id": compute_energy_from_logits(
            id_test_logits,
            temperature=ENERGY_T
        ),
        "ood": compute_energy_from_logits(
            ood_test_logits,
            temperature=ENERGY_T
        )
    }

    fixed_test_scores[f"KNN-k{KNN_K}"] = {
        "id": compute_knn_scores(
            knn=knn,
            features=id_test_features,
            k=KNN_K
        ),
        "ood": compute_knn_scores(
            knn=knn,
            features=ood_test_features,
            k=KNN_K
        )
    }

    fixed_test_scores[f"ViM-dim{vim_stats['vim_dim']}"] = {
        "id": compute_vim_scores(
            features=id_test_features,
            logits=id_test_logits,
            vim_stats=vim_stats
        ),
        "ood": compute_vim_scores(
            features=ood_test_features,
            logits=ood_test_logits,
            vim_stats=vim_stats
        )
    }

    print("Computing ODIN ID/OOD test scores. This may take longer.")

    fixed_test_scores[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = {
        "id": compute_odin_scores(
            model=model,
            data_loader=id_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        ),
        "ood": compute_odin_scores(
            model=model,
            data_loader=ood_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )
    }

    # -----------------------------------------------------
    # Step 4. Run contaminated calibration experiments
    # -----------------------------------------------------
    print("\n[Step 4] Run IQR adaptive calibration experiments")

    all_results = []

    for contamination_ratio in CONTAMINATION_RATIOS:
        print("\n" + "=" * 120)
        print(f"Contamination ratio: {contamination_ratio * 100:.1f}%")
        print("=" * 120)

        contaminated_calib_dataset, num_id, num_ood = create_contaminated_calibration_dataset(
            clean_calib_dataset=clean_calib_dataset,
            svhn_train_dataset=svhn_train_dataset,
            contamination_ratio=contamination_ratio,
            total_calib_size=CALIB_SIZE,
            seed=42 + int(contamination_ratio * 10000)
        )

        contaminated_calib_loader = DataLoader(
            contaminated_calib_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        print(f"Calibration ID samples : {num_id}")
        print(f"Calibration OOD samples: {num_ood}")

        calib_logits, calib_features, _ = extract_logits_and_features(
            model,
            contaminated_calib_loader
        )

        calib_scores_dict = {}

        calib_scores_dict["MSP"] = compute_msp_from_logits(calib_logits)

        calib_scores_dict["Energy"] = compute_energy_from_logits(
            calib_logits,
            temperature=ENERGY_T
        )

        calib_scores_dict[f"KNN-k{KNN_K}"] = compute_knn_scores(
            knn=knn,
            features=calib_features,
            k=KNN_K
        )

        calib_scores_dict[f"ViM-dim{vim_stats['vim_dim']}"] = compute_vim_scores(
            features=calib_features,
            logits=calib_logits,
            vim_stats=vim_stats
        )

        print("Computing ODIN calibration scores. This may take longer.")

        calib_scores_dict[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = compute_odin_scores(
            model=model,
            data_loader=contaminated_calib_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )

        # Evaluate standard and IQR adaptive calibration.
        for method_name, calib_scores in calib_scores_dict.items():

            # Standard calibration.
            standard_result = evaluate_standard(
                method_name=method_name,
                calib_scores=calib_scores,
                id_test_scores=fixed_test_scores[method_name]["id"],
                ood_test_scores=fixed_test_scores[method_name]["ood"]
            )

            standard_row = {
                "contamination_ratio": float(contamination_ratio),
                "contamination_percent": float(contamination_ratio * 100),
                "num_calib_id": int(num_id),
                "num_calib_ood": int(num_ood),
                **standard_result
            }

            all_results.append(standard_row)

            print(
                f"{method_name:<28} | "
                f"calib=standard     | "
                f"lambda=NA  | "
                f"removed={0.0:>6.2f}% | "
                f"threshold={standard_result['threshold']:>10.6f} | "
                f"AUROC={standard_result['auroc_percent']:>6.2f}% | "
                f"FPR95={standard_result['fpr95_percent']:>6.2f}% | "
                f"ID_TPR={standard_result['id_tpr_percent']:>6.2f}%"
            )

            # IQR adaptive calibration for each lambda.
            for lambda_iqr in IQR_LAMBDAS:
                iqr_result = evaluate_iqr_adaptive(
                    method_name=method_name,
                    calib_scores=calib_scores,
                    id_test_scores=fixed_test_scores[method_name]["id"],
                    ood_test_scores=fixed_test_scores[method_name]["ood"],
                    lambda_iqr=lambda_iqr
                )

                iqr_row = {
                    "contamination_ratio": float(contamination_ratio),
                    "contamination_percent": float(contamination_ratio * 100),
                    "num_calib_id": int(num_id),
                    "num_calib_ood": int(num_ood),
                    **iqr_result
                }

                all_results.append(iqr_row)

                print(
                    f"{method_name:<28} | "
                    f"calib=iqr_adaptive | "
                    f"lambda={lambda_iqr:<3.1f} | "
                    f"removed={iqr_result['removed_percent']:>6.2f}% | "
                    f"threshold={iqr_result['threshold']:>10.6f} | "
                    f"AUROC={iqr_result['auroc_percent']:>6.2f}% | "
                    f"FPR95={iqr_result['fpr95_percent']:>6.2f}% | "
                    f"ID_TPR={iqr_result['id_tpr_percent']:>6.2f}%"
                )

    # -----------------------------------------------------
    # Step 5. Save full results
    # -----------------------------------------------------
    results_df = pd.DataFrame(all_results)

    full_csv_path = os.path.join(
        RESULT_DIR,
        "iqr_adaptive_calibration_full_results.csv"
    )

    full_npy_path = os.path.join(
        RESULT_DIR,
        "iqr_adaptive_calibration_full_results.npy"
    )

    results_df.to_csv(full_csv_path, index=False)
    np.save(full_npy_path, all_results, allow_pickle=True)

    print("\nSaved full IQR adaptive results to:")
    print(full_csv_path)
    print(full_npy_path)

    # -----------------------------------------------------
    # Step 6. Save compact FPR95 tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("FPR95 (%) tables")
    print("=" * 100)

    # Standard table.
    standard_df = results_df[results_df["calibration_type"] == "standard"]

    standard_fpr_table = standard_df.pivot(
        index="method",
        columns="contamination_percent",
        values="fpr95_percent"
    )

    standard_fpr_path = os.path.join(
        RESULT_DIR,
        "iqr_standard_fpr95_table.csv"
    )

    print("\nStandard calibration FPR95")
    print(standard_fpr_table.round(2))
    standard_fpr_table.to_csv(standard_fpr_path)

    # IQR tables by lambda.
    for lambda_iqr in IQR_LAMBDAS:
        subset = results_df[
            (results_df["calibration_type"] == "iqr_adaptive") &
            (results_df["lambda_iqr"] == lambda_iqr)
        ]

        fpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="fpr95_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"iqr_lambda_{str(lambda_iqr).replace('.', '_')}_fpr95_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"IQR adaptive FPR95 | lambda={lambda_iqr}")
        print("-" * 100)
        print(fpr_table.round(2))
        fpr_table.to_csv(path)

    # -----------------------------------------------------
    # Step 7. Save ID TPR tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("ID TPR (%) tables")
    print("=" * 100)

    standard_id_tpr_table = standard_df.pivot(
        index="method",
        columns="contamination_percent",
        values="id_tpr_percent"
    )

    standard_id_tpr_path = os.path.join(
        RESULT_DIR,
        "iqr_standard_id_tpr_table.csv"
    )

    print("\nStandard calibration ID TPR")
    print(standard_id_tpr_table.round(2))
    standard_id_tpr_table.to_csv(standard_id_tpr_path)

    for lambda_iqr in IQR_LAMBDAS:
        subset = results_df[
            (results_df["calibration_type"] == "iqr_adaptive") &
            (results_df["lambda_iqr"] == lambda_iqr)
        ]

        id_tpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="id_tpr_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"iqr_lambda_{str(lambda_iqr).replace('.', '_')}_id_tpr_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"IQR adaptive ID TPR | lambda={lambda_iqr}")
        print("-" * 100)
        print(id_tpr_table.round(2))
        id_tpr_table.to_csv(path)

    # -----------------------------------------------------
    # Step 8. Save removed ratio tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("Removed calibration samples (%) tables")
    print("=" * 100)

    for lambda_iqr in IQR_LAMBDAS:
        subset = results_df[
            (results_df["calibration_type"] == "iqr_adaptive") &
            (results_df["lambda_iqr"] == lambda_iqr)
        ]

        removed_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="removed_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"iqr_lambda_{str(lambda_iqr).replace('.', '_')}_removed_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"IQR adaptive removed ratio | lambda={lambda_iqr}")
        print("-" * 100)
        print(removed_table.round(2))
        removed_table.to_csv(path)

    # -----------------------------------------------------
    # Step 9. Save plots
    # -----------------------------------------------------
    print("\n[Step 9] Save plots")

    save_iqr_fpr95_plots(results_df, RESULT_DIR)
    save_iqr_id_tpr_plots(results_df, RESULT_DIR)
    save_iqr_removed_ratio_plots(results_df, RESULT_DIR)

    print("\nIQR adaptive trimming experiment completed.")


if __name__ == "__main__":
    main()

Using device: cuda
Loaded model from: ./results/cifar10_resnet18_clean.pth

[Step 1] Extract fixed train/test logits and features
Train features shape   : (45000, 512)
ID test features shape : (10000, 512)
OOD test features shape: (26032, 512)

[Step 2] Fit KNN and ViM
KNN k: 50
ViM dim: 256

[Step 3] Precompute fixed ID/OOD test scores
Computing ODIN ID/OOD test scores. This may take longer.

[Step 4] Run IQR adaptive calibration experiments

Contamination ratio: 0.0%
Calibration ID samples : 5000
Calibration OOD samples: 0
Computing ODIN calibration scores. This may take longer.
MSP                          | calib=standard     | lambda=NA  | removed=  0.00% | threshold=  0.774561 | AUROC= 90.77% | FPR95= 59.37% | ID_TPR= 94.72%
MSP                          | calib=iqr_adaptive | lambda=0.5 | removed= 23.12% | threshold=  0.998621 | AUROC= 90.77% | FPR95=  3.97% | ID_TPR= 72.65%
MSP                          | calib=iqr_adaptive | lambda=1.0 | removed= 21.54% | threshold=  0.998224 | 

# Adaptive Trimming - GMM

In [ ]:
"""
GMM-based Adaptive Trimming Calibration for Post-hoc OOD Detection

Goal:
- Use a pretrained CIFAR-10 classifier.
- Contaminate only the calibration set with SVHN samples.
- Compare standard calibration with GMM-based adaptive trimming.
- Evaluate FPR95, ID TPR, ID rejection rate, and estimated contamination ratio.

Dataset protocol:
- ID dataset        : CIFAR-10
- OOD dataset       : SVHN
- Train feature bank: CIFAR-10 train split only
- Clean calibration : CIFAR-10 calibration split only
- Contamination OOD : SVHN train split
- ID test           : CIFAR-10 test
- OOD test          : SVHN test

OOD methods:
1. MSP
2. Energy
3. KNN
4. ViM
5. ODIN

Calibration methods:
1. Standard calibration:
   - Use all calibration scores.
   - Threshold = 5th percentile of calibration scores.

2. GMM adaptive trimming:
   - Fit a 2-component Gaussian Mixture Model on calibration scores.
   - Identify the low-mean component as the OOD-like component.
   - Remove samples whose posterior probability of belonging to the low-mean component
     is greater than or equal to a given posterior threshold.
   - Compute the 5th percentile threshold from the remaining scores.

Score convention:
- Higher score = more ID-like
- Lower score  = more OOD-like

Decision rule:
- score >= threshold -> ID
- score < threshold  -> OOD
"""

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset

from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture

import matplotlib.pyplot as plt


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


# =========================================================
# 2. Configuration
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./data"
RESULT_DIR = "./results"

MODEL_PATH = "./results/cifar10_resnet18_clean.pth"

BATCH_SIZE = 128
NUM_CLASSES = 10

# These values must match the clean baseline training split.
TRAIN_SIZE = 45000
CALIB_SIZE = 5000

# OOD method hyperparameters.
ENERGY_T = 1.0
KNN_K = 50
VIM_DIM = 256

# ODIN hyperparameters.
ODIN_T = 1000.0
ODIN_EPSILON = 0.0014

# Calibration contamination ratios.
CONTAMINATION_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20]

# GMM posterior probability thresholds.
# Smaller values remove more samples.
# Larger values are more conservative.
GMM_POSTERIOR_THRESHOLDS = [0.5, 0.7, 0.9]

os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")


# =========================================================
# 3. Dataset and transforms
# =========================================================

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# CIFAR-10 train data is used to reconstruct the same train/calibration split.
cifar10_full_train = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=test_transform
)

train_dataset, clean_calib_dataset = random_split(
    cifar10_full_train,
    [TRAIN_SIZE, CALIB_SIZE],
    generator=torch.Generator().manual_seed(42)
)

id_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

svhn_train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=test_transform
)

ood_test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

id_test_loader = DataLoader(
    id_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ood_test_loader = DataLoader(
    ood_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 4. Model
# =========================================================

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10.

    This wrapper exposes:
    - logits for MSP, Energy, and ODIN
    - penultimate features for KNN and ViM
    """

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = resnet18(weights=None)

        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        backbone.maxpool = nn.Identity()

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool
        )

        feature_dim = backbone.fc.in_features
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward_features(self, x):
        """
        Extract penultimate features before the classifier head.
        """
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        return features

    def forward(self, x, return_features=False):
        """
        Forward pass.

        If return_features=True, return both logits and features.
        """
        features = self.forward_features(x)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits


model = CIFARResNet18(NUM_CLASSES).to(DEVICE)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}\n"
        "Please run the clean baseline training script first."
    )

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 5. Contaminated calibration dataset
# =========================================================

def create_contaminated_calibration_dataset(
    clean_calib_dataset,
    svhn_train_dataset,
    contamination_ratio,
    total_calib_size=5000,
    seed=42
):
    """
    Create a contaminated calibration dataset.

    The total calibration size is fixed.

    Example:
    - contamination_ratio = 0.05
    - total_calib_size = 5000
    - ID samples  = 4750
    - OOD samples = 250
    """
    rng = np.random.default_rng(seed)

    num_ood = int(total_calib_size * contamination_ratio)
    num_id = total_calib_size - num_ood

    id_indices = rng.choice(
        len(clean_calib_dataset),
        size=num_id,
        replace=False
    )

    id_subset = Subset(clean_calib_dataset, id_indices.tolist())

    if num_ood > 0:
        ood_indices = rng.choice(
            len(svhn_train_dataset),
            size=num_ood,
            replace=False
        )

        ood_subset = Subset(svhn_train_dataset, ood_indices.tolist())
        contaminated_dataset = ConcatDataset([id_subset, ood_subset])
    else:
        contaminated_dataset = id_subset

    return contaminated_dataset, num_id, num_ood


# =========================================================
# 6. Logit and feature extraction
# =========================================================

@torch.no_grad()
def extract_logits_and_features(model, data_loader):
    """
    Extract logits and penultimate features from a dataloader.
    """
    model.eval()

    all_logits = []
    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(DEVICE)

        logits, features = model(images, return_features=True)

        all_logits.append(logits.detach().cpu())
        all_features.append(features.detach().cpu())
        all_labels.append(torch.as_tensor(labels))

    logits = torch.cat(all_logits, dim=0).numpy()
    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    return logits, features, labels


# =========================================================
# 7. OOD score functions
# =========================================================

def compute_msp_from_logits(logits):
    """
    Compute Maximum Softmax Probability.

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    probs = torch.softmax(logits_tensor, dim=1)
    scores = probs.max(dim=1).values.numpy()
    return scores


def compute_energy_from_logits(logits, temperature=1.0):
    """
    Compute Energy-based ID scores.

    This returns -E(x):
        score = T * logsumexp(logits / T)

    Higher score means more ID-like.
    """
    logits_tensor = torch.from_numpy(logits)
    scores = temperature * torch.logsumexp(
        logits_tensor / temperature,
        dim=1
    ).numpy()
    return scores


def fit_knn_feature_bank(train_features, k=50):
    """
    Fit KNN on clean ID training features.
    """
    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=-1
    )
    knn.fit(train_features)
    return knn


def compute_knn_scores(knn, features, k=50):
    """
    Compute KNN-based ID scores.

    Score = negative distance to the k-th nearest ID training feature.
    Higher score means more ID-like.
    """
    distances, _ = knn.kneighbors(features, n_neighbors=k)
    kth_distance = distances[:, -1]
    scores = -kth_distance
    return scores


def fit_vim(train_features, train_logits, vim_dim=256):
    """
    Fit simplified ViM statistics using clean ID training data.
    """
    feature_mean = np.mean(train_features, axis=0, keepdims=True)

    centered_features = train_features - feature_mean

    _, _, vt = np.linalg.svd(centered_features, full_matrices=False)

    feature_dim = train_features.shape[1]
    vim_dim = min(vim_dim, feature_dim - 1)

    residual_space = vt[vim_dim:].T

    train_residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    train_max_logit = np.max(train_logits, axis=1)

    alpha = np.mean(train_max_logit) / (np.mean(train_residual_norm) + 1e-12)

    vim_stats = {
        "feature_mean": feature_mean,
        "residual_space": residual_space,
        "alpha": alpha,
        "vim_dim": vim_dim
    }

    return vim_stats


def compute_vim_scores(features, logits, vim_stats):
    """
    Compute simplified ViM ID scores.

    score = max_original_logit - alpha * residual_norm

    Higher score means more ID-like.
    """
    feature_mean = vim_stats["feature_mean"]
    residual_space = vim_stats["residual_space"]
    alpha = vim_stats["alpha"]

    centered_features = features - feature_mean

    residual_norm = np.linalg.norm(
        np.matmul(centered_features, residual_space),
        axis=1
    )

    virtual_logit = alpha * residual_norm
    max_original_logit = np.max(logits, axis=1)

    scores = max_original_logit - virtual_logit
    return scores


def compute_odin_scores(model, data_loader, temperature=1000.0, epsilon=0.0014):
    """
    Compute ODIN scores.

    Higher score means more ID-like.
    """
    model.eval()

    all_scores = []

    for images, _ in data_loader:
        images = images.to(DEVICE)
        images.requires_grad_(True)

        logits = model(images)
        scaled_logits = logits / temperature

        pseudo_labels = scaled_logits.argmax(dim=1)
        loss = F.cross_entropy(scaled_logits, pseudo_labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = torch.sign(images.grad.data)

        perturbed_images = images - epsilon * gradient_sign
        perturbed_images = perturbed_images.detach()

        with torch.no_grad():
            perturbed_logits = model(perturbed_images)
            perturbed_scaled_logits = perturbed_logits / temperature
            probs = torch.softmax(perturbed_scaled_logits, dim=1)

            scores = probs.max(dim=1).values

        all_scores.append(scores.detach().cpu())

    scores = torch.cat(all_scores, dim=0).numpy()
    return scores


# =========================================================
# 8. Threshold and metric functions
# =========================================================

def get_standard_threshold(calib_scores):
    """
    Standard calibration threshold.

    Threshold = 5th percentile of all calibration scores.
    """
    return np.percentile(calib_scores, 5)


def get_gmm_adaptive_threshold(
    calib_scores,
    posterior_threshold=0.5,
    random_state=42
):
    """
    GMM-based adaptive trimming threshold.

    Procedure:
    1. Fit a 2-component GMM on 1D calibration scores.
    2. Identify the low-mean component as the OOD-like component.
    3. Remove samples whose posterior probability of belonging to the low-mean
       component is >= posterior_threshold.
    4. Compute the 5th percentile threshold from the remaining samples.

    Important:
    - The low-component weight can be interpreted as an approximate estimated
      contamination ratio.
    - This is not guaranteed to match the true contamination ratio exactly.
    """
    calib_scores = np.asarray(calib_scores).reshape(-1, 1)

    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=random_state,
        reg_covar=1e-6,
        n_init=5
    )

    gmm.fit(calib_scores)

    means = gmm.means_.reshape(-1)
    weights = gmm.weights_.reshape(-1)

    # The lower-mean component is treated as the OOD-like component.
    low_component = int(np.argmin(means))
    high_component = int(np.argmax(means))

    posterior = gmm.predict_proba(calib_scores)
    low_component_prob = posterior[:, low_component]

    remove_mask = low_component_prob >= posterior_threshold
    keep_mask = ~remove_mask

    original_scores = calib_scores.reshape(-1)

    kept_scores = original_scores[keep_mask]
    removed_scores = original_scores[remove_mask]

    # Safety fallback:
    # If GMM removes all samples, fall back to standard calibration.
    if len(kept_scores) == 0:
        kept_scores = original_scores
        remove_mask = np.zeros_like(original_scores, dtype=bool)
        keep_mask = ~remove_mask
        removed_scores = np.array([])

    threshold = np.percentile(kept_scores, 5)

    removed_ratio = len(removed_scores) / len(original_scores)

    stats = {
        "low_component": low_component,
        "high_component": high_component,
        "low_component_mean": float(means[low_component]),
        "high_component_mean": float(means[high_component]),
        "low_component_weight": float(weights[low_component]),
        "high_component_weight": float(weights[high_component]),
        "estimated_contamination_percent": float(weights[low_component] * 100),
        "posterior_threshold": float(posterior_threshold),
        "num_removed": int(len(removed_scores)),
        "removed_ratio": float(removed_ratio),
        "removed_percent": float(removed_ratio * 100),
        "num_kept": int(len(kept_scores))
    }

    return threshold, stats


def compute_fpr95(ood_scores, threshold):
    """
    Compute FPR95.

    FPR95 is the fraction of OOD samples incorrectly accepted as ID.
    """
    return np.mean(ood_scores >= threshold)


def compute_id_tpr(id_scores, threshold):
    """
    Compute ID TPR.

    ID TPR is the fraction of ID samples correctly accepted as ID.
    """
    return np.mean(id_scores >= threshold)


def compute_id_rejection_rate(id_scores, threshold):
    """
    Compute ID rejection rate.

    ID rejection rate is the fraction of ID samples incorrectly rejected as OOD.
    """
    return np.mean(id_scores < threshold)


def compute_auroc(id_scores, ood_scores):
    """
    Compute AUROC.

    ID label  = 1
    OOD label = 0
    """
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores))
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores
    ])

    return roc_auc_score(y_true, y_score)


def evaluate_standard(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores
):
    """
    Evaluate standard calibration.
    """
    threshold = get_standard_threshold(calib_scores)

    auroc = compute_auroc(id_test_scores, ood_test_scores)
    fpr95 = compute_fpr95(ood_test_scores, threshold)
    id_tpr = compute_id_tpr(id_test_scores, threshold)
    id_reject = compute_id_rejection_rate(id_test_scores, threshold)

    return {
        "method": method_name,
        "calibration_type": "standard",
        "posterior_threshold": np.nan,
        "threshold": float(threshold),

        "auroc": float(auroc),
        "auroc_percent": float(auroc * 100),

        "fpr95": float(fpr95),
        "fpr95_percent": float(fpr95 * 100),

        "id_tpr": float(id_tpr),
        "id_tpr_percent": float(id_tpr * 100),

        "id_rejection_rate": float(id_reject),
        "id_rejection_percent": float(id_reject * 100),

        "low_component": np.nan,
        "high_component": np.nan,
        "low_component_mean": np.nan,
        "high_component_mean": np.nan,
        "low_component_weight": np.nan,
        "high_component_weight": np.nan,
        "estimated_contamination_percent": np.nan,
        "num_removed": 0,
        "removed_ratio": 0.0,
        "removed_percent": 0.0,
        "num_kept": len(calib_scores)
    }


def evaluate_gmm_adaptive(
    method_name,
    calib_scores,
    id_test_scores,
    ood_test_scores,
    posterior_threshold,
    random_state=42
):
    """
    Evaluate GMM-based adaptive trimming calibration.
    """
    threshold, stats = get_gmm_adaptive_threshold(
        calib_scores=calib_scores,
        posterior_threshold=posterior_threshold,
        random_state=random_state
    )

    auroc = compute_auroc(id_test_scores, ood_test_scores)
    fpr95 = compute_fpr95(ood_test_scores, threshold)
    id_tpr = compute_id_tpr(id_test_scores, threshold)
    id_reject = compute_id_rejection_rate(id_test_scores, threshold)

    return {
        "method": method_name,
        "calibration_type": "gmm_adaptive",
        "posterior_threshold": float(posterior_threshold),
        "threshold": float(threshold),

        "auroc": float(auroc),
        "auroc_percent": float(auroc * 100),

        "fpr95": float(fpr95),
        "fpr95_percent": float(fpr95 * 100),

        "id_tpr": float(id_tpr),
        "id_tpr_percent": float(id_tpr * 100),

        "id_rejection_rate": float(id_reject),
        "id_rejection_percent": float(id_reject * 100),

        **stats
    }


# =========================================================
# 9. Plotting functions
# =========================================================

def save_gmm_fpr95_plots(results_df, result_dir):
    """
    Save FPR95 vs contamination plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        standard_df = subset[subset["calibration_type"] == "standard"].sort_values(
            by="contamination_percent"
        )

        plt.plot(
            standard_df["contamination_percent"],
            standard_df["fpr95_percent"],
            marker="o",
            label="standard"
        )

        for pth in sorted(subset["posterior_threshold"].dropna().unique()):
            gmm_df = subset[
                (subset["calibration_type"] == "gmm_adaptive") &
                (subset["posterior_threshold"] == pth)
            ].sort_values(by="contamination_percent")

            plt.plot(
                gmm_df["contamination_percent"],
                gmm_df["fpr95_percent"],
                marker="o",
                label=f"GMM p≥{pth}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("FPR95 (%)")
        plt.title(f"GMM adaptive calibration: FPR95 | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"gmm_fpr95_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_gmm_id_tpr_plots(results_df, result_dir):
    """
    Save ID TPR vs contamination plots for each OOD method.
    """
    for method in sorted(results_df["method"].unique()):
        subset = results_df[results_df["method"] == method]

        plt.figure(figsize=(8, 6))

        standard_df = subset[subset["calibration_type"] == "standard"].sort_values(
            by="contamination_percent"
        )

        plt.plot(
            standard_df["contamination_percent"],
            standard_df["id_tpr_percent"],
            marker="o",
            label="standard"
        )

        for pth in sorted(subset["posterior_threshold"].dropna().unique()):
            gmm_df = subset[
                (subset["calibration_type"] == "gmm_adaptive") &
                (subset["posterior_threshold"] == pth)
            ].sort_values(by="contamination_percent")

            plt.plot(
                gmm_df["contamination_percent"],
                gmm_df["id_tpr_percent"],
                marker="o",
                label=f"GMM p≥{pth}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("ID TPR (%)")
        plt.title(f"GMM adaptive calibration: ID TPR | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"gmm_id_tpr_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_gmm_removed_ratio_plots(results_df, result_dir):
    """
    Save removed ratio vs contamination plots for each OOD method.
    """
    gmm_only_df = results_df[results_df["calibration_type"] == "gmm_adaptive"]

    for method in sorted(gmm_only_df["method"].unique()):
        subset = gmm_only_df[gmm_only_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for pth in sorted(subset["posterior_threshold"].unique()):
            pth_df = subset[subset["posterior_threshold"] == pth].sort_values(
                by="contamination_percent"
            )

            plt.plot(
                pth_df["contamination_percent"],
                pth_df["removed_percent"],
                marker="o",
                label=f"GMM p≥{pth}"
            )

        plt.xlabel("Calibration contamination ratio (%)")
        plt.ylabel("Removed calibration samples (%)")
        plt.title(f"GMM removed ratio | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(result_dir, f"gmm_removed_ratio_{safe_name}.png")

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


def save_gmm_estimated_contamination_plots(results_df, result_dir):
    """
    Save estimated contamination ratio vs true contamination ratio plots.
    """
    gmm_only_df = results_df[results_df["calibration_type"] == "gmm_adaptive"]

    for method in sorted(gmm_only_df["method"].unique()):
        subset = gmm_only_df[gmm_only_df["method"] == method]

        plt.figure(figsize=(8, 6))

        for pth in sorted(subset["posterior_threshold"].unique()):
            pth_df = subset[subset["posterior_threshold"] == pth].sort_values(
                by="contamination_percent"
            )

            plt.plot(
                pth_df["contamination_percent"],
                pth_df["estimated_contamination_percent"],
                marker="o",
                label=f"GMM p≥{pth}"
            )

        plt.plot(
            [0, 20],
            [0, 20],
            linestyle="--",
            label="ideal estimate"
        )

        plt.xlabel("True calibration contamination ratio (%)")
        plt.ylabel("GMM low-component weight (%)")
        plt.title(f"GMM estimated contamination | {method}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_name = method.replace("/", "_").replace(" ", "_").replace(".", "_")
        save_path = os.path.join(
            result_dir,
            f"gmm_estimated_contamination_{safe_name}.png"
        )

        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"Saved plot: {save_path}")


# =========================================================
# 10. Main experiment
# =========================================================

def main():
    """
    Run the full GMM adaptive trimming experiment.
    """

    # -----------------------------------------------------
    # Step 1. Extract fixed train/test logits and features
    # -----------------------------------------------------
    print("\n[Step 1] Extract fixed train/test logits and features")

    train_logits, train_features, _ = extract_logits_and_features(
        model,
        train_loader
    )

    id_test_logits, id_test_features, _ = extract_logits_and_features(
        model,
        id_test_loader
    )

    ood_test_logits, ood_test_features, _ = extract_logits_and_features(
        model,
        ood_test_loader
    )

    print(f"Train features shape   : {train_features.shape}")
    print(f"ID test features shape : {id_test_features.shape}")
    print(f"OOD test features shape: {ood_test_features.shape}")

    # -----------------------------------------------------
    # Step 2. Fit KNN and ViM
    # -----------------------------------------------------
    print("\n[Step 2] Fit KNN and ViM")

    knn = fit_knn_feature_bank(
        train_features=train_features,
        k=KNN_K
    )

    vim_stats = fit_vim(
        train_features=train_features,
        train_logits=train_logits,
        vim_dim=VIM_DIM
    )

    print(f"KNN k: {KNN_K}")
    print(f"ViM dim: {vim_stats['vim_dim']}")

    # -----------------------------------------------------
    # Step 3. Precompute fixed ID/OOD test scores
    # -----------------------------------------------------
    print("\n[Step 3] Precompute fixed ID/OOD test scores")

    fixed_test_scores = {}

    fixed_test_scores["MSP"] = {
        "id": compute_msp_from_logits(id_test_logits),
        "ood": compute_msp_from_logits(ood_test_logits)
    }

    fixed_test_scores["Energy"] = {
        "id": compute_energy_from_logits(
            id_test_logits,
            temperature=ENERGY_T
        ),
        "ood": compute_energy_from_logits(
            ood_test_logits,
            temperature=ENERGY_T
        )
    }

    fixed_test_scores[f"KNN-k{KNN_K}"] = {
        "id": compute_knn_scores(
            knn=knn,
            features=id_test_features,
            k=KNN_K
        ),
        "ood": compute_knn_scores(
            knn=knn,
            features=ood_test_features,
            k=KNN_K
        )
    }

    fixed_test_scores[f"ViM-dim{vim_stats['vim_dim']}"] = {
        "id": compute_vim_scores(
            features=id_test_features,
            logits=id_test_logits,
            vim_stats=vim_stats
        ),
        "ood": compute_vim_scores(
            features=ood_test_features,
            logits=ood_test_logits,
            vim_stats=vim_stats
        )
    }

    print("Computing ODIN ID/OOD test scores. This may take longer.")

    fixed_test_scores[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = {
        "id": compute_odin_scores(
            model=model,
            data_loader=id_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        ),
        "ood": compute_odin_scores(
            model=model,
            data_loader=ood_test_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )
    }

    # -----------------------------------------------------
    # Step 4. Run contaminated calibration experiments
    # -----------------------------------------------------
    print("\n[Step 4] Run GMM adaptive calibration experiments")

    all_results = []

    for contamination_ratio in CONTAMINATION_RATIOS:
        print("\n" + "=" * 120)
        print(f"Contamination ratio: {contamination_ratio * 100:.1f}%")
        print("=" * 120)

        contaminated_calib_dataset, num_id, num_ood = create_contaminated_calibration_dataset(
            clean_calib_dataset=clean_calib_dataset,
            svhn_train_dataset=svhn_train_dataset,
            contamination_ratio=contamination_ratio,
            total_calib_size=CALIB_SIZE,
            seed=42 + int(contamination_ratio * 10000)
        )

        contaminated_calib_loader = DataLoader(
            contaminated_calib_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        print(f"Calibration ID samples : {num_id}")
        print(f"Calibration OOD samples: {num_ood}")

        calib_logits, calib_features, _ = extract_logits_and_features(
            model,
            contaminated_calib_loader
        )

        calib_scores_dict = {}

        calib_scores_dict["MSP"] = compute_msp_from_logits(calib_logits)

        calib_scores_dict["Energy"] = compute_energy_from_logits(
            calib_logits,
            temperature=ENERGY_T
        )

        calib_scores_dict[f"KNN-k{KNN_K}"] = compute_knn_scores(
            knn=knn,
            features=calib_features,
            k=KNN_K
        )

        calib_scores_dict[f"ViM-dim{vim_stats['vim_dim']}"] = compute_vim_scores(
            features=calib_features,
            logits=calib_logits,
            vim_stats=vim_stats
        )

        print("Computing ODIN calibration scores. This may take longer.")

        calib_scores_dict[f"ODIN-T{ODIN_T}-eps{ODIN_EPSILON}"] = compute_odin_scores(
            model=model,
            data_loader=contaminated_calib_loader,
            temperature=ODIN_T,
            epsilon=ODIN_EPSILON
        )

        # Evaluate standard and GMM adaptive calibration.
        for method_name, calib_scores in calib_scores_dict.items():

            # Standard calibration.
            standard_result = evaluate_standard(
                method_name=method_name,
                calib_scores=calib_scores,
                id_test_scores=fixed_test_scores[method_name]["id"],
                ood_test_scores=fixed_test_scores[method_name]["ood"]
            )

            standard_row = {
                "contamination_ratio": float(contamination_ratio),
                "contamination_percent": float(contamination_ratio * 100),
                "num_calib_id": int(num_id),
                "num_calib_ood": int(num_ood),
                **standard_result
            }

            all_results.append(standard_row)

            print(
                f"{method_name:<28} | "
                f"calib=standard     | "
                f"p=NA  | "
                f"removed={0.0:>6.2f}% | "
                f"est_contam=NA     | "
                f"threshold={standard_result['threshold']:>10.6f} | "
                f"FPR95={standard_result['fpr95_percent']:>6.2f}% | "
                f"ID_TPR={standard_result['id_tpr_percent']:>6.2f}%"
            )

            # GMM adaptive calibration.
            for posterior_threshold in GMM_POSTERIOR_THRESHOLDS:
                gmm_result = evaluate_gmm_adaptive(
                    method_name=method_name,
                    calib_scores=calib_scores,
                    id_test_scores=fixed_test_scores[method_name]["id"],
                    ood_test_scores=fixed_test_scores[method_name]["ood"],
                    posterior_threshold=posterior_threshold,
                    random_state=42
                )

                gmm_row = {
                    "contamination_ratio": float(contamination_ratio),
                    "contamination_percent": float(contamination_ratio * 100),
                    "num_calib_id": int(num_id),
                    "num_calib_ood": int(num_ood),
                    **gmm_result
                }

                all_results.append(gmm_row)

                print(
                    f"{method_name:<28} | "
                    f"calib=gmm_adaptive | "
                    f"p={posterior_threshold:<3.1f} | "
                    f"removed={gmm_result['removed_percent']:>6.2f}% | "
                    f"est_contam={gmm_result['estimated_contamination_percent']:>6.2f}% | "
                    f"threshold={gmm_result['threshold']:>10.6f} | "
                    f"FPR95={gmm_result['fpr95_percent']:>6.2f}% | "
                    f"ID_TPR={gmm_result['id_tpr_percent']:>6.2f}%"
                )

    # -----------------------------------------------------
    # Step 5. Save full results
    # -----------------------------------------------------
    results_df = pd.DataFrame(all_results)

    full_csv_path = os.path.join(
        RESULT_DIR,
        "gmm_adaptive_calibration_full_results.csv"
    )

    full_npy_path = os.path.join(
        RESULT_DIR,
        "gmm_adaptive_calibration_full_results.npy"
    )

    results_df.to_csv(full_csv_path, index=False)
    np.save(full_npy_path, all_results, allow_pickle=True)

    print("\nSaved full GMM adaptive results to:")
    print(full_csv_path)
    print(full_npy_path)

    # -----------------------------------------------------
    # Step 6. Save compact FPR95 tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("FPR95 (%) tables")
    print("=" * 100)

    standard_df = results_df[results_df["calibration_type"] == "standard"]

    standard_fpr_table = standard_df.pivot(
        index="method",
        columns="contamination_percent",
        values="fpr95_percent"
    )

    standard_fpr_path = os.path.join(
        RESULT_DIR,
        "gmm_standard_fpr95_table.csv"
    )

    print("\nStandard calibration FPR95")
    print(standard_fpr_table.round(2))
    standard_fpr_table.to_csv(standard_fpr_path)

    for posterior_threshold in GMM_POSTERIOR_THRESHOLDS:
        subset = results_df[
            (results_df["calibration_type"] == "gmm_adaptive") &
            (results_df["posterior_threshold"] == posterior_threshold)
        ]

        fpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="fpr95_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"gmm_posterior_{str(posterior_threshold).replace('.', '_')}_fpr95_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"GMM adaptive FPR95 | posterior threshold={posterior_threshold}")
        print("-" * 100)
        print(fpr_table.round(2))
        fpr_table.to_csv(path)

    # -----------------------------------------------------
    # Step 7. Save ID TPR tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("ID TPR (%) tables")
    print("=" * 100)

    standard_id_tpr_table = standard_df.pivot(
        index="method",
        columns="contamination_percent",
        values="id_tpr_percent"
    )

    standard_id_tpr_path = os.path.join(
        RESULT_DIR,
        "gmm_standard_id_tpr_table.csv"
    )

    print("\nStandard calibration ID TPR")
    print(standard_id_tpr_table.round(2))
    standard_id_tpr_table.to_csv(standard_id_tpr_path)

    for posterior_threshold in GMM_POSTERIOR_THRESHOLDS:
        subset = results_df[
            (results_df["calibration_type"] == "gmm_adaptive") &
            (results_df["posterior_threshold"] == posterior_threshold)
        ]

        id_tpr_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="id_tpr_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"gmm_posterior_{str(posterior_threshold).replace('.', '_')}_id_tpr_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"GMM adaptive ID TPR | posterior threshold={posterior_threshold}")
        print("-" * 100)
        print(id_tpr_table.round(2))
        id_tpr_table.to_csv(path)

    # -----------------------------------------------------
    # Step 8. Save removed ratio tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("Removed calibration samples (%) tables")
    print("=" * 100)

    for posterior_threshold in GMM_POSTERIOR_THRESHOLDS:
        subset = results_df[
            (results_df["calibration_type"] == "gmm_adaptive") &
            (results_df["posterior_threshold"] == posterior_threshold)
        ]

        removed_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="removed_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"gmm_posterior_{str(posterior_threshold).replace('.', '_')}_removed_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"GMM adaptive removed ratio | posterior threshold={posterior_threshold}")
        print("-" * 100)
        print(removed_table.round(2))
        removed_table.to_csv(path)

    # -----------------------------------------------------
    # Step 9. Save estimated contamination tables
    # -----------------------------------------------------
    print("\n" + "=" * 100)
    print("Estimated contamination (%) tables")
    print("=" * 100)

    for posterior_threshold in GMM_POSTERIOR_THRESHOLDS:
        subset = results_df[
            (results_df["calibration_type"] == "gmm_adaptive") &
            (results_df["posterior_threshold"] == posterior_threshold)
        ]

        estimated_table = subset.pivot(
            index="method",
            columns="contamination_percent",
            values="estimated_contamination_percent"
        )

        path = os.path.join(
            RESULT_DIR,
            f"gmm_posterior_{str(posterior_threshold).replace('.', '_')}_estimated_contamination_table.csv"
        )

        print("\n" + "-" * 100)
        print(f"GMM estimated contamination | posterior threshold={posterior_threshold}")
        print("-" * 100)
        print(estimated_table.round(2))
        estimated_table.to_csv(path)

    # -----------------------------------------------------
    # Step 10. Save plots
    # -----------------------------------------------------
    print("\n[Step 10] Save plots")

    save_gmm_fpr95_plots(results_df, RESULT_DIR)
    save_gmm_id_tpr_plots(results_df, RESULT_DIR)
    save_gmm_removed_ratio_plots(results_df, RESULT_DIR)
    save_gmm_estimated_contamination_plots(results_df, RESULT_DIR)

    print("\nGMM adaptive trimming experiment completed.")


if __name__ == "__main__":
    main()

Using device: cuda
Loaded model from: ./results/cifar10_resnet18_clean.pth

[Step 1] Extract fixed train/test logits and features
Train features shape   : (45000, 512)
ID test features shape : (10000, 512)
OOD test features shape: (26032, 512)

[Step 2] Fit KNN and ViM
KNN k: 50
ViM dim: 256

[Step 3] Precompute fixed ID/OOD test scores
Computing ODIN ID/OOD test scores. This may take longer.

[Step 4] Run GMM adaptive calibration experiments

Contamination ratio: 0.0%
Calibration ID samples : 5000
Calibration OOD samples: 0
Computing ODIN calibration scores. This may take longer.
MSP                          | calib=standard     | p=NA  | removed=  0.00% | est_contam=NA     | threshold=  0.774561 | FPR95= 59.37% | ID_TPR= 94.72%
MSP                          | calib=gmm_adaptive | p=0.5 | removed= 20.66% | est_contam= 20.88% | threshold=  0.997856 | FPR95=  5.44% | ID_TPR= 74.91%
MSP                          | calib=gmm_adaptive | p=0.7 | removed= 20.40% | est_contam= 20.88% | threshol